# Function Calling — LLM jako "mózg" który wywołuje funkcje

## Jak deweloperzy budują AI asystentów i agenty?

### Plan warsztatu

1. **Kalkulator + pierwszy Function Call**: Zbudujemy narzędzie i od razu zobaczymy je w akcji!
2. **Instructor**: Poznamy jak zmusić LLM-a do odpowiadania w ścisłym formacie Pydantic
3. **Pogoda i prezydenci**: Dodamy poważniejsze narzędzia — z prawdziwym API i bazą danych
4. **LLM wybiera**: Zobaczmy jak LLM sam decyduje którego narzędzia użyć
5. **Wikipedia i web search**: Podłączymy prawdziwe źródła wiedzy
6. **Pętla agentowa**: Zbudujemy mini-agenta który sam decyduje jakie narzędzia użyć

### Czym jest Function Calling?

Normalnie LLM dostaje pytanie i generuje tekst. Ale co jeśli pytanie wymaga:
- Sprawdzenia **aktualnej** pogody?
- Wykonania **obliczeń** matematycznych?
- Przeszukania **Wikipedii** albo **internetu**?

LLM nie ma do tego dostępu — ale może **poprosić** nasz program o wywołanie odpowiedniej funkcji.
To jest właśnie **Function Calling** (wywoływanie funkcji):

```
Użytkownik: "Jaka jest pogoda w Krakowie?"
    ↓
LLM myśli: "Potrzebuję funkcji get_weather z argumentem city='Kraków'"
    ↓
Nasz program: wywołuje get_weather("Kraków") → "18°C, słonecznie"
    ↓
LLM: "W Krakowie jest 18°C i słonecznie!"
```

**Tak działają ChatGPT, Claude, GitHub Copilot** — LLM + zestaw narzędzi + pętla decyzyjna.

## 1. Konfiguracja środowiska

In [1]:
from utils import ensure_package

ensure_package("openai")
ensure_package("pydantic")
ensure_package("instructor")
ensure_package("wikipedia")
ensure_package("ddgs")

from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from IPython.display import display, Markdown
import requests
import json
import math
import urllib.parse
from datetime import datetime
from pathlib import Path

print("Pakiety załadowane!")

Pakiety załadowane!


## 2. Połączenie z LLM-em

Poniższa komórka automatycznie szuka działającego LLM-a. Zanim ją uruchomisz:

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:12px; border-radius:4px;">

**Upewnij się, że LLM jest uruchomiony!**

- **LM Studio** (macOS / Windows / Linux) — otwórz aplikację → załaduj model → zakładka *Local Server* → zielony przycisk *Start Server*
- **Ollama** — otwórz terminal i wpisz `ollama serve`
  - *macOS:* jeśli zainstalowałeś przez `.dmg`, Ollama zwykle startuje automatycznie (ikonka w pasku menu)
  - *Windows:* Ollama działa jako usługa po instalacji — jeśli nie, uruchom ręcznie z menu Start
  - *Linux:* `ollama serve` w terminalu (lub `systemctl start ollama` jeśli zainstalowano przez `curl`)

Szczegóły instalacji: patrz `setup_local_llm.ipynb` lub `docs/LOKALNE_LLM.md`
</div>

**Function calling działa najlepiej z modelami:** `qwen3.5:4b+`, `gemma4:4b+`, `qwen3:8b+`

In [2]:
from utils import connect_llm, extract_reasoning, print_reasoning, clean_content

# --- TEST: bezpośrednie połączenie na port 4242 z modelem gemma-4-e4b ---
from openai import OpenAI
import instructor

client = OpenAI(base_url="http://localhost:4242/v1", api_key="lm-studio")
instructor_client = instructor.from_openai(client, mode=instructor.Mode.MD_JSON)
MODEL_NAME = [m.id for m in client.models.list().data if "gemma" in m.id.lower()][0]

print(f"\nKlient LLM gotowy!  Model: {MODEL_NAME}")
print(f"Instructor:         {'tak' if instructor_client else 'nie'}")



Klient LLM gotowy!  Model: gemma-4-e4b-it-mlx
Instructor:         tak


In [3]:
# ── System prompt — wspólny dla wszystkich funkcji ──────────────────
# Definiujemy go raz, w jednym miejscu. Dzięki temu:
#   1. Łatwo zmienić zachowanie WSZYSTKICH funkcji naraz
#   2. Studenci widzą, że system prompt to konfiguracja, nie magia
#   3. Unikamy copy-paste tego samego tekstu w 7 miejscach

_BASE_SYSTEM_PROMPT = (
    "Jesteś pomocnym asystentem. Odpowiadaj po polsku. "
    "ZAWSZE używaj dostępnych narzędzi gdy pytanie tego dotyczy — "
    "nie próbuj odpowiadać z pamięci."
)

# <|think|> włącza natywny tok myślenia w Gemma-4 — inne modele go nie potrzebują
_is_gemma = "gemma" in MODEL_NAME.lower()
DEFAULT_SYSTEM_PROMPT = f"<|think|>{_BASE_SYSTEM_PROMPT}" if _is_gemma else _BASE_SYSTEM_PROMPT

print(f"System prompt ({len(DEFAULT_SYSTEM_PROMPT)} znaków):")
print(f"  {DEFAULT_SYSTEM_PROMPT[:80]}...")
if _is_gemma:
    print("💡 Gemma wykryta → dodano <|think|> (natywne myślenie)")

System prompt (152 znaków):
  <|think|>Jesteś pomocnym asystentem. Odpowiadaj po polsku. ZAWSZE używaj dostępn...
💡 Gemma wykryta → dodano <|think|> (natywne myślenie)


## 3. Kalkulator — twoje pierwsze narzędzie + Function Call!

Zacznijmy od **najprostszego** możliwego narzędzia — kalkulatora.
Zobaczymy **cały cykl Function Calling** krok po kroku.

In [4]:
def calculate(expression: str) -> str:
    """
    Wykonuje obliczenie matematyczne.

    Args:
        expression: Wyrażenie matematyczne, np. '2 + 2', 'sqrt(144)', '15 * 7'
    """
    try:
        # LLM często pisze ^ zamiast ** (notacja matematyczna vs Python)
        expression = expression.replace("^", "**")
        allowed = {"sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,
                   "pi": math.pi, "abs": abs, "round": round, "pow": pow}
        result = eval(expression, {"__builtins__": {}}, allowed)
        return f"{expression} = {result}"
    except Exception as e:
        return f"Błąd w obliczeniu '{expression}': {e}"


print("Test:")
print(calculate("sqrt(144) + 7 * 3"))
print(calculate("2 ** 10"))

Test:
sqrt(144) + 7 * 3 = 33.0
2 ** 10 = 1024


### Pierwszy Function Call — pod maską!

Mamy narzędzie. Teraz **opiszemy** je dla LLM-a (w formacie JSON Schema)
i wyślemy pytanie. LLM **sam zdecyduje** czy potrzebuje kalkulatora.

Zobaczmy **dokładnie** co się dzieje na każdym etapie:

In [5]:
# Opis narzędzia dla LLM-a (JSON Schema):
calc_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Wykonuje obliczenie matematyczne. Użyj gdy trzeba coś policzyć.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Wyrażenie matematyczne, np. '2+2', 'sqrt(144)'"
                }
            },
            "required": ["expression"]
        }
    }
}

def calculate_function_call(user_prompt):
    """Pełny cykl Function Calling: pytanie → LLM → narzędzie → odpowiedź."""
    if not client:
        print("LLM niedostępny — uruchom LM Studio lub Ollamę.")
        return

    W = 60
    def box(text):
        print("╔" + "═"*W + "╗")
        print("║  " + text.ljust(W-4) + "  ║")
        print("╚" + "═"*W + "╝")

    box("KROK 1: Wysyłamy pytanie + opis narzędzia do LLM-a")
    print(f"\n  Pytanie: \"{user_prompt}\"")
    print(f"  Narzędzie: calculate — {calc_tool['function']['description']}")
    print(f"  → Wysyłam do {MODEL_NAME}...\n")

    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME, messages=messages, tools=[calc_tool], temperature=0.1
    )

    msg = response.choices[0].message

    print_reasoning(msg)

    content = clean_content(msg)
    if content and msg.tool_calls:
        print(f"  💬 LLM mówi: {content[:300]}")
        print()

    if msg.tool_calls:
        tc = msg.tool_calls[0]
        func_name = tc.function.name
        func_args = json.loads(tc.function.arguments)

        box("KROK 2: LLM wybrał narzędzie!")
        print(f"  Narzędzie: {func_name}")
        print(f"  Argumenty: {func_args}")

        result = calculate(**func_args)

        print()
        box("KROK 3: Wywołujemy funkcję i odsyłamy wynik do LLM-a")
        print(f"  Wynik: {result}")

        messages.append(msg)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        final = client.chat.completions.create(model=MODEL_NAME, messages=messages, temperature=0.1)

        print()
        box("KROK 4: LLM formułuje ostateczną odpowiedź")
        print()
        display(Markdown(clean_content(final.choices[0].message)))
    else:
        print(f"  LLM odpowiedział bez narzędzia: {(content or '')[:200]}")

# --- Uruchomienie ---
calculate_function_call("Ile to jest 17 razy 23?")

╔════════════════════════════════════════════════════════════╗
║  KROK 1: Wysyłamy pytanie + opis narzędzia do LLM-a        ║
╚════════════════════════════════════════════════════════════╝

  Pytanie: "Ile to jest 17 razy 23?"
  Narzędzie: calculate — Wykonuje obliczenie matematyczne. Użyj gdy trzeba coś policzyć.
  → Wysyłam do gemma-4-e4b-it-mlx...



  🧠 Tok myślenia (reasoning):
     The user is asking for the result of a multiplication: 17 times 23. I should use the `calculate` tool to perform this arithmetic operation.

╔════════════════════════════════════════════════════════════╗
║  KROK 2: LLM wybrał narzędzie!                             ║
╚════════════════════════════════════════════════════════════╝
  Narzędzie: calculate
  Argumenty: {'expression': '17 * 23'}

╔════════════════════════════════════════════════════════════╗
║  KROK 3: Wywołujemy funkcję i odsyłamy wynik do LLM-a      ║
╚════════════════════════════════════════════════════════════╝
  Wynik: 17 * 23 = 391



╔════════════════════════════════════════════════════════════╗
║  KROK 4: LLM formułuje ostateczną odpowiedź                ║
╚════════════════════════════════════════════════════════════╝



17 razy 23 to **391**.

<div style="background:#d4edda; border-left:4px solid #28a745; padding:14px; border-radius:4px;">

**To jest cały cykl Function Calling!**

```
Użytkownik: "Ile to jest 17 razy 23?"
    ↓
LLM myśli: "Potrzebuję calculate(expression='17 * 23')"     ← LLM generuje argumenty
    ↓
Nasz kod: calculate("17 * 23") → {"result": 391.0}          ← Python liczy
    ↓
LLM: "17 razy 23 to 391"                                     ← LLM formułuje odpowiedź
```

Kluczowe: **LLM nie liczy sam** — prosi NASZ kod o obliczenie. My kontrolujemy narzędzia.

</div>

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; border-radius:4px; margin-top:10px;">

**🧠 Tok myślenia (reasoning):** Modele mogą pokazywać swój wewnętrzny tok myślenia na różne sposoby:
Qwen3 → osobne pole `reasoning_content`, Gemma-4 → tokeny `<|channel>thought` w odpowiedzi.
Nasza funkcja `extract_reasoning()` z `utils.py` wyciąga reasoning niezależnie od formatu.

</div>

Teraz dodamy więcej narzędzi — pogodę i bazę prezydentów.

Ale najpierw — **zauważyłeś, że `calc_tool` to ręcznie pisany JSON Schema?**
Był poręczny do nauki — widać dokładnie co LLM dostaje.
Ale pisanie tego ręcznie dla każdego narzędzia to błędogenna robota.

**Pydantic** umie wygenerować ten sam schemat automatycznie!

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; border-radius:4px;">
<b>Pydantic w jednym zdaniu:</b> Definiujesz klasę z polami i typami → Python automatycznie
waliduje dane i generuje schemat JSON — ten sam format którego używają OpenAI, Anthropic, Ollama.
</div>

In [6]:
# Przypomnijmy — ręczny JSON Schema naszego kalkulatora:
print("RĘCZNIE napisany schemat (calc_tool → parameters):")
print(json.dumps(calc_tool["function"]["parameters"], indent=2, ensure_ascii=False))

# A teraz — Pydantic generuje to samo z klasy!
class CalculateArgs(BaseModel):
    expression: str = Field(..., description="Wyrażenie matematyczne, np. '2+2', 'sqrt(144)'")

print("\nPYDANTIC wygenerowany schemat (CalculateArgs.model_json_schema()):")
print(json.dumps(CalculateArgs.model_json_schema(), indent=2, ensure_ascii=False))

# Jedyna różnica: Pydantic dodaje "title" — reszta identyczna!
# LLM ignoruje title, więc to nie ma znaczenia.

RĘCZNIE napisany schemat (calc_tool → parameters):
{
  "type": "object",
  "properties": {
    "expression": {
      "type": "string",
      "description": "Wyrażenie matematyczne, np. '2+2', 'sqrt(144)'"
    }
  },
  "required": [
    "expression"
  ]
}

PYDANTIC wygenerowany schemat (CalculateArgs.model_json_schema()):
{
  "properties": {
    "expression": {
      "description": "Wyrażenie matematyczne, np. '2+2', 'sqrt(144)'",
      "title": "Expression",
      "type": "string"
    }
  },
  "required": [
    "expression"
  ],
  "title": "CalculateArgs",
  "type": "object"
}


In [7]:
# Helper — generuje definicję narzędzia z modelu Pydantic.
# Nigdy więcej ręcznego JSON Schema!

def make_tool(name, description, args_model):
    """Tworzy definicję narzędzia FC z modelu Pydantic — zero ręcznego JSON Schema!"""
    schema = args_model.model_json_schema()
    schema.pop("title", None)  # OpenAI nie potrzebuje tytułu klasy
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": schema,
        }
    }

# Test — porównajmy ręczny calc_tool z automatycznym:
calc_tool_auto = make_tool(
    "calculate",
    "Wykonuje obliczenie matematyczne. Użyj gdy trzeba coś policzyć.",
    CalculateArgs,
)

print("Ręczny calc_tool:")
print(json.dumps(calc_tool["function"]["parameters"], indent=2))
print("\nAutomatyczny (make_tool + Pydantic):")
print(json.dumps(calc_tool_auto["function"]["parameters"], indent=2))
print("\nOd teraz używamy make_tool() — Pydantic pisze JSON Schema za nas!")

Ręczny calc_tool:
{
  "type": "object",
  "properties": {
    "expression": {
      "type": "string",
      "description": "Wyra\u017cenie matematyczne, np. '2+2', 'sqrt(144)'"
    }
  },
  "required": [
    "expression"
  ]
}

Automatyczny (make_tool + Pydantic):
{
  "properties": {
    "expression": {
      "description": "Wyra\u017cenie matematyczne, np. '2+2', 'sqrt(144)'",
      "title": "Expression",
      "type": "string"
    }
  },
  "required": [
    "expression"
  ],
  "type": "object"
}

Od teraz używamy make_tool() — Pydantic pisze JSON Schema za nas!


<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:12px; border-radius:4px;">

**Podsumowanie: JSON Schema na dwa sposoby**

| | Ręcznie | Pydantic |
|---|---|---|
| **Jak** | Piszesz `{"type": "object", "properties": {...}}` | Piszesz `class Args(BaseModel): ...` |
| **Schema** | Ty go tworzysz | `model_json_schema()` generuje automatycznie |
| **Błędy** | Łatwo o literówkę | Pydantic waliduje za Ciebie |
| **Kiedy** | Żeby zrozumieć co siedzi pod spodem | W produkcji — zawsze |

Od tej pory będziemy definiować narzędzia przez **modele Pydantic + `make_tool()`**.

</div>

<div style="background:#e8f4f8; border-left:4px solid #17a2b8; padding:12px; border-radius:4px; margin-top:10px;">

**💡 Dlaczego jawne `make_tool()`, a nie magia?**

W produkcji nikt nie pisze `make_tool()` ręcznie — biblioteki takie jak **LangChain** potrafią
wyciągnąć nazwę, opis i schemat automatycznie z docstringa i type hints:

```python
# LangChain — dekorator @tool robi wszystko za Ciebie:
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """Sprawdza aktualną pogodę w podanym mieście."""
    ...
```

My celowo **pokazujemy bebechy** — żebyście widzieli co dokładnie LLM dostaje od naszego kodu.
Dzięki temu w ćwiczeniu 1 będziecie mogli celowo zmienić `description` i zobaczyć jak to wpływa
na zachowanie modelu. Z magicznym dekoratorem nie byłoby tego widać.

</div>

### 3b. *(opcjonalne)* Instructor — Structured Output

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; border-radius:4px;">
Ta sekcja pokazuje <b>drugi</b> sposób użycia Pydantic z LLM-em. Nie jest wymagana do dalszej pracy
z Function Calling — możesz ją pominąć i wrócić później.
</div>

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:12px; border-radius:4px; margin-top:8px;">
⏱️ <b>Uwaga:</b> Komórki w tej sekcji mogą wykonywać się <b>20–60 sekund</b> (zależnie od modelu).
Instructor wymusza format odpowiedzi przez prompt engineering — LLM musi wygenerować poprawny JSON,
a jeśli się pomyli, instructor powtarza zapytanie. To wolniejsze niż zwykły Function Calling.<br><br>
Jeśli korzystacie z <b>jednego serwera prowadzącego</b> — zapytania obsługiwane są jedno po drugim (kolejka).
Przy 20 osobach i ~30s na zapytanie, ostatnia osoba w kolejce może czekać nawet kilka minut.
Cierpliwości! 🙂
</div>

W Function Calling Pydantic opisuje **wejście** narzędzia (argumenty).
Ale jest też odwrotny kierunek: zmusić LLM-a żeby **odpowiedział** w formacie Pydantic.

Biblioteka **`instructor`** opakowuje klienta OpenAI i dodaje parametr `response_model=`.

```
Bez instructor:   LLM → "Wrocław to miasto na Dolnym Śląsku, ma ok. 670 tys. mieszkańców..."  (tekst)
Z instructor:     LLM → CityInfo(name="Wrocław", population=670000, region="Dolny Śląsk")   (obiekt!)
```

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; border-radius:4px;">
<b>Instructor pod maską:</b>
<ol>
<li>Bierze Twój model Pydantic → generuje JSON Schema</li>
<li>Wysyła schema + prompt do LLM-a: "Odpowiedz w tym formacie JSON"</li>
<li>Parsuje odpowiedź LLM-a przez Pydantic</li>
<li>Jeśli walidacja się nie powiedzie → wysyła LLM-owi feedback i prosi o poprawę (do 3 prób!)</li>
</ol>
</div>

In [8]:
# Prosty model — informacja o mieście
class CityInfo(BaseModel):
    name: str = Field(..., description="Nazwa miasta")
    country: str = Field(..., description="Kraj")
    population_approx: int = Field(..., description="Przybliżona liczba mieszkańców")
    famous_for: str = Field(..., description="Z czego miasto jest znane (1 zdanie)")

# instructor w akcji — LLM MUSI odpowiedzieć jako CityInfo!
if instructor_client:
    city = instructor_client.chat.completions.create(
        model=MODEL_NAME,
        response_model=CityInfo,
        messages=[{"role": "user", "content": "Opowiedz o Wrocławiu"}],
    )
    print(f"Typ wyniku: {type(city).__name__}")
    print(f"Surowy obiekt: {city}")
    print(f"\nJSON:\n{city.model_dump_json(indent=2)}")
else:
    print("instructor_client niedostępny — uruchom LLM-a i wróć do sekcji 2.")

Typ wyniku: CityInfo
Surowy obiekt: name='Wrocław' country='Polska' population_approx=670000 famous_for='Jest znany ze swojego urokliwego Starego Miasta, licznych mostów oraz krasnali.'

JSON:
{
  "name": "Wrocław",
  "country": "Polska",
  "population_approx": 670000,
  "famous_for": "Jest znany ze swojego urokliwego Starego Miasta, licznych mostów oraz krasnali."
}


In [9]:
# ↓ To NIE jest tekst — to obiekt Pythona! Mamy dostęp do pól: ↓
if instructor_client:
    print(f"city.name              → {city.name}")
    print(f"city.country           → {city.country}")
    print(f"city.population_approx → {city.population_approx}")
    print(f"city.famous_for        → {city.famous_for}")
    print(f"\nPopulacja to int — można liczyć: {city.population_approx * 2} = podwójna populacja")

city.name              → Wrocław
city.country           → Polska
city.population_approx → 670000
city.famous_for        → Jest znany ze swojego urokliwego Starego Miasta, licznych mostów oraz krasnali.

Populacja to int — można liczyć: 1340000 = podwójna populacja


#### Co się stanie gdy pytanie nie pasuje do modelu?

LLM **musi** wypełnić wszystkie pola — nawet gdy pytanie dotyczy czegoś zupełnie innego.
Zobaczmy co się stanie gdy zapytamy o smak lodów, a model wymaga danych o mieście:

In [10]:
# MaybeCityInfo — model z polem "found" na wypadek pytań nie w temacie

class MaybeCityInfo(BaseModel):
    found: bool = Field(..., description="Czy pytanie dotyczy konkretnego miasta?")
    name: str = Field("", description="Nazwa miasta (puste jeśli found=False)")
    country: str = Field("", description="Kraj (puste jeśli found=False)")
    population_approx: int = Field(0, description="Przybliżona liczba mieszkańców (0 jeśli found=False)")
    famous_for: str = Field("", description="Z czego miasto jest znane (puste jeśli found=False)")
    error_message: str = Field("", description="Wyjaśnienie dlaczego nie znaleziono miasta (puste jeśli found=True)")

test_questions = [
    "Opowiedz o Gdańsku",
    "Opowiedz o Chrząszczyżewoszycach",
    "Jaki jest najlepszy smak lodów?",
    "Ile nóg ma pająk?",
]

if instructor_client:
    for q in test_questions:
        print(f"\nPytanie: \"{q}\"")
        result = instructor_client.chat.completions.create(
            model=MODEL_NAME,
            response_model=MaybeCityInfo,
            messages=[{"role": "user", "content": q}],
        )
        if result.found:
            print(f"  ✓ {result.name} ({result.country}), {result.population_approx:_} mieszkańców")
            print(f"    Znane z: {result.famous_for}")
        else:
            print(f"  ✗ result.found=False → LLM: {result.error_message}")
else:
    print("instructor_client niedostępny.")


Pytanie: "Opowiedz o Gdańsku"


  ✓ Gdańsk (Polska), 480_000 mieszkańców
    Znane z: Stare Miasto, historyczne porty, bursztyn

Pytanie: "Opowiedz o Chrząszczyżewoszycach"


  ✗ result.found=False → LLM: Nie znaleziono informacji o mieście o nazwie Chrząszczyżewoszyce.

Pytanie: "Jaki jest najlepszy smak lodów?"


  ✗ result.found=False → LLM: Pytanie nie dotyczy konkretnego miasta.

Pytanie: "Ile nóg ma pająk?"


  ✗ result.found=False → LLM: Pytanie nie dotyczy konkretnego miasta.


#### Mechanizm retry — instructor sam naprawia błędy

Jedną z największych zalet `instructor` jest **automatyczny retry z feedbackiem**.
Jeśli LLM zwróci niepoprawny JSON, instructor:
1. Łapie błąd parsowania/walidacji
2. **Odsyła go do LLM-a** jako wiadomość: *"Popraw odpowiedź, oto co było źle: ..."*
3. LLM poprawia się — i tak do `max_retries` prób

Zobaczmy to na żywo! Przechwytujemy odpowiedź LLM-a i **celowo psujemy JSON**
zanim instructor go zobaczy. Instructor wykryje błąd i poprosi LLM-a o poprawę:

In [11]:
# --- Sabotaż! Przechwytujemy odpowiedź LLM-a i psujemy JSON ---
_orig_create = client.chat.completions.create
_call_count = [0]

def _sabotage(*args, **kwargs):
    _call_count[0] += 1

    # Przy retry — pokazujemy CO instructor wysyła do LLM-a
    if _call_count[0] == 2:
        msgs = kwargs.get('messages', [])
        print("  🔍 Co instructor wysyła do LLM-a przy retry:")
        print("  " + "─" * 55)
        for m in msgs:
            role = m.get('role', '?')
            content = str(m.get('content', ''))
            if len(content) > 300:
                content = content[:300] + "..."
            print(f"  [{role}] {content}\n")
        print("  " + "─" * 55 + "\n")

    resp = _orig_create(*args, **kwargs)
    content = resp.choices[0].message.content

    if _call_count[0] == 1:  # tylko pierwszą odpowiedź psujemy
        print(f"  📡 Próba 1 — oryginalna odpowiedź LLM-a:")
        print(f"      {content[:200]}\n")

        broken = content.replace("}", "###ZEPSUTE###")
        print(f"  💥 Psujemy JSON przed oddaniem instructorowi:")
        print(f"      {broken[:200]}\n")
        resp.choices[0].message.content = broken
    else:
        print(f"  📡 Próba {_call_count[0]} — LLM poprawił odpowiedź:")
        print(f"      {content[:200]}\n")
    return resp

client.chat.completions.create = _sabotage

# Tworzymy nowego instructor_client z "popsutym" klientem
import instructor
_sabotaged_ic = instructor.from_openai(client, mode=instructor.Mode.MD_JSON)

class CityRetryDemo(BaseModel):
    name: str = Field(..., description="Nazwa miasta")
    country: str = Field(..., description="Kraj")
    population_approx: int = Field(..., description="Przybliżona liczba mieszkańców")

print("🔬 Demo: psujemy JSON + podglądamy retry!\n")

city = _sabotaged_ic.chat.completions.create(
    model=MODEL_NAME,
    response_model=CityRetryDemo,
    messages=[{"role": "user", "content": "Opowiedz o Wrocławiu"}],
    max_retries=3,
)
print(f"  ✅ Sukces! {city.name}, {city.country}, {city.population_approx:,}")

# Przywracamy oryginalny client.create (ważne dla dalszych komórek!)
client.chat.completions.create = _orig_create

🔬 Demo: psujemy JSON + podglądamy retry!



  📡 Próba 1 — oryginalna odpowiedź LLM-a:
      ```json
{
  "name": "Wrocław",
  "country": "Polska",
  "population_approx": 670000
}
```

  💥 Psujemy JSON przed oddaniem instructorowi:
      ```json
{
  "name": "Wrocław",
  "country": "Polska",
  "population_approx": 670000
###ZEPSUTE###
```

  🔍 Co instructor wysyła do LLM-a przy retry:
  ───────────────────────────────────────────────────────
  [system] 
        Parse the content and return a JSON object matching this schema:

        {
  "properties": {
    "name": {
      "description": "Nazwa miasta",
      "title": "Name",
      "type": "string"
    },
    "country": {
      "description": "Kraj",
      "title": "Country",
      "type": "string...

  [user] Opowiedz o Wrocławiu

Return the correct JSON response within a ```json codeblock. not the JSON_SCHEMA

  [assistant] ```json
{
  "name": "Wrocław",
  "country": "Polska",
  "population_approx": 670000
###ZEPSUTE###
```

  [user] Correct your JSON ONLY RESPONSE, based on the 

  📡 Próba 2 — LLM poprawił odpowiedź:
      ```json
{
  "name": "Wrocław",
  "country": "Polska",
  "population_approx": 670000
}
```

  ✅ Sukces! Wrocław, Polska, 670,000


<div style="background:#d4edda; border-left:4px solid #28a745; padding:12px; border-radius:4px;">
<b>Co się właśnie stało?</b>
<ol>
<li><b>[system]</b> — Instructor wstrzyknął JSON Schema do system message</li>
<li><b>[user]</b> — Nasze pytanie + instrukcja "Return JSON in codeblock"</li>
<li><b>[assistant]</b> — Zepsuta odpowiedź (instructor "widzi" ją jako oryginalną odpowiedź LLM-a)</li>
<li><b>[user]</b> — <code>"Correct your JSON ONLY RESPONSE, based on the following errors:"</code> + pełny traceback!</li>
<li>LLM czyta feedback i <b>sam się poprawia</b></li>
</ol>
W prawdziwym użyciu nie musisz nic psuć — instructor robi to <b>automatycznie</b>
gdy LLM zwróci niepoprawny format. To Twoja siatka bezpieczeństwa! 🛡️
</div>

#### Function Calling vs. Structured Output — dwa różne mechanizmy!

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:14px; border-radius:4px;">

| | **Function Calling** (`client`) | **Structured Output** (`instructor_client`) |
|---|---|---|
| **Co robi LLM** | **Wybiera** którą funkcję Pythona wywołać | **Wypełnia** schemat Pydantic danymi |
| **Kto wykonuje pracę** | Nasz kod Pythona (np. łączy się z API pogody) | Sam LLM (generuje dane "z głowy") |
| **Parametr** | `tools=tools_definition` | `response_model=MyModel` |
| **Wynik** | LLM woła funkcję → wynik → LLM formułuje tekst | Obiekt Pydantic (np. `CityInfo(...)`) |

**Analogia kuchenna:**
- **Function Calling** = LLM jest **kelnerem** — przyjmuje zamówienie i mówi kucharzowi (naszemu kodowi) co ugotować
- **Structured Output** = LLM jest **kucharzem** który musi podać danie dokładnie wg karty — nazwa, składniki, czas przygotowania, każde pole wypełnione, zero improwizacji

</div>

## 4. Pogoda — prawdziwe API + fallback

Kalkulator pokazał nam cały cykl FC. Teraz zbudujemy **poważniejsze narzędzie**
— pobierające prawdziwą pogodę z API [wttr.in](https://wttr.in).

Jeśli API jest niedostępne — automatycznie przełącza się na dane zastępcze (mock).
Narzędzie zwraca **zwykły tekst** — LLM sobie z nim poradzi.

In [12]:
MOCK_WEATHER = {
    "Kraków":   {"temp": 18, "opis": "słonecznie",              "wilgotność": 45},
    "Warszawa": {"temp": 15, "opis": "pochmurno",               "wilgotność": 70},
    "Gdańsk":   {"temp": 12, "opis": "deszcz",                  "wilgotność": 85},
    "Wrocław":  {"temp": 20, "opis": "słonecznie",              "wilgotność": 40},
    "Poznań":   {"temp": 16, "opis": "częściowe zachmurzenie",  "wilgotność": 55},
}


def get_weather(city: str) -> str:
    """
    Sprawdza aktualną pogodę w podanym mieście.
    Pobiera dane z wttr.in; jeśli API niedostępne — używa danych zastępczych.

    Args:
        city: Nazwa miasta, np. 'Kraków', 'Warszawa', 'Gdańsk'
    """
    # Próba pobrania prawdziwych danych
    try:
        url = f"https://wttr.in/{urllib.parse.quote(city)}?format=j1"
        r = requests.get(url, timeout=5, headers={"Accept-Language": "pl"})
        r.raise_for_status()
        data = r.json()
        current = data["current_condition"][0]
        desc_list = current.get("lang_pl", current.get("weatherDesc", [{"value": "brak danych"}]))
        temp = current["temp_C"]
        cond = desc_list[0]["value"]
        hum = current["humidity"]
        obs_time = current.get("localObsDateTime", "")
        time_info = f", dane z {obs_time}" if obs_time else ""
        return f"{city}: {temp}°C, {cond}, wilgotność {hum}%{time_info} (wttr.in)"
    except Exception:
        pass

    # Fallback — dane mockowane
    if city in MOCK_WEATHER:
        m = MOCK_WEATHER[city]
        fake_time = datetime.now().strftime("%Y-%m-%d %H:%M")
        return f"{city}: {m['temp']}°C, {m['opis']}, wilgotność {m['wilgotność']}%, dane z {fake_time} (dane zastępcze)"

    return f"Brak danych pogodowych dla: {city}"


# Test:
print("Test pogody:")
print(get_weather("Wrocław"))

Test pogody:
Wrocław: 19°C, Słonecznie, wilgotność 43%, dane z 2026-05-10 07:52 PM (wttr.in)


## 5. Baza prezydentów Polski

Trzecie narzędzie — baza danych o prezydentach III RP wczytywana z pliku `prezydenci_polski.md`.

In [13]:
def load_presidents():
    """Ładuje dane o prezydentach z pliku prezydenci_polski.md"""
    md_path = Path("prezydenci_polski.md")
    if not md_path.exists():
        return []

    text = md_path.read_text(encoding="utf-8")
    presidents = []
    current = {}

    for line in text.split("\n"):
        line = line.strip()
        if line.startswith("### ") and not line.startswith("####"):
            if current:
                presidents.append(current)
            current = {"imię": line[4:].strip()}
        elif line.startswith("- **") and ":**" in line:
            key = line.split("**")[1].rstrip(":")
            value = line.split(":** ", 1)[1] if ":** " in line else ""
            current[key.lower()] = value

    if current:
        presidents.append(current)
    return presidents

PREZYDENCI = load_presidents()


def search_presidents(query: str) -> str:
    """
    Przeszukuje bazę danych o prezydentach Polski (III RP).

    Każdy prezydent ma pola: Kadencja, Lata życia, Miejsce urodzenia,
    Wiek w dniu objęcia urzędu, Partia, Wykształcenie, Poprzedni urząd,
    Kluczowe wydarzenia, Mało znany fakt, Długość kadencji.

    WAŻNE: Używaj KRÓTKICH zapytań — najlepiej samo nazwisko (np. 'Kwaśniewski')
    lub jedno–dwa słowa kluczowe (np. 'wykształcenie', 'najmłodszy').
    Nie pisz pełnych zdań — wyszukiwarka działa na dopasowanie słów.

    Args:
        query: Krótkie zapytanie, np. 'Kwaśniewski', 'wykształcenie', 'mało znany fakt'
    """
    if not PREZYDENCI:
        return "Brak danych — nie znaleziono pliku prezydenci_polski.md"

    query_lower = query.lower()
    words = query_lower.split()

    # Scoring — liczymy ile słów pasuje do każdego prezydenta
    scored = []
    for p in PREZYDENCI:
        all_text = " ".join(f"{k} {v}" for k, v in p.items()).lower()
        hits = sum(1 for w in words if w in all_text)
        if hits > 0:
            scored.append((hits / len(words), p))

    # Sortujemy po trafności (malejąco)
    scored.sort(key=lambda x: x[0], reverse=True)

    # Zwracamy prezydentów z trafnością >= 50% (lub najlepszego jeśli nikt nie osiąga 50%)
    wyniki = []
    for score, p in scored:
        if score >= 0.5 or (not wyniki and score > 0):
            lines = [f"### {p.get('imię', '?')}"]
            for key, val in p.items():
                if key != 'imię':
                    lines.append(f"  - {key}: {val}")
            wyniki.append("\n".join(lines))

    if wyniki:
        return "\n\n".join(wyniki)
    return f"Nie znaleziono wyników dla zapytania: {query}"


AVAILABLE_TOOLS = {
    "get_weather": get_weather,
    "calculate": calculate,
    "search_presidents": search_presidents,
}

print(f"Załadowano {len(PREZYDENCI)} prezydentów z pliku .md")
print(f"Mamy {len(AVAILABLE_TOOLS)} narzędzia: {list(AVAILABLE_TOOLS.keys())}")

Załadowano 7 prezydentów z pliku .md
Mamy 3 narzędzia: ['get_weather', 'calculate', 'search_presidents']


## 6. Trzy narzędzia — LLM wybiera!

Mamy kalkulator, pogodę i prezydentów. Teraz złóżmy je razem.

W sekcji 3 zdefiniowaliśmy `make_tool()` i `CalculateArgs` — teraz dodamy
modele Pydantic dla pozostałych narzędzi i sklejmy wszystko razem.

In [14]:
# Modele Pydantic dla argumentów każdego narzędzia
# (CalculateArgs już mamy z sekcji 3)

class GetWeatherArgs(BaseModel):
    city: str = Field(..., description="Nazwa miasta, np. 'Kraków', 'Warszawa'")

class SearchPresidentsArgs(BaseModel):
    query: str = Field(..., description="Pytanie o prezydentów, np. 'kto rządził najdłużej', 'mało znane fakty', 'Kwaśniewski'")

# Składamy tools_definition za pomocą make_tool()
tools_definition = [
    make_tool("get_weather",
              "Sprawdza aktualną pogodę w podanym mieście (temperatura, opis, wilgotność, godzina pomiaru).",
              GetWeatherArgs),
    make_tool("calculate",
              "Wykonuje obliczenie matematyczne. Użyj gdy trzeba coś policzyć.",
              CalculateArgs),
    make_tool("search_presidents",
              "Przeszukuje bazę prezydentów Polski (III RP). "
              "Pola: kadencja, wiek w dniu objęcia urzędu, partia, wykształcenie, kluczowe wydarzenia, mało znane fakty. "
              "Używaj KRÓTKICH zapytań — nazwisko lub słowo kluczowe (np. 'Kwaśniewski', 'wykształcenie').",
              SearchPresidentsArgs),
]

print("LLM ma teraz 3 narzędzia do wyboru:")
for t in tools_definition:
    print(f"  {t['function']['name']}: {t['function']['description'][:60]}...")

print("\nPrzykładowa definicja (get_weather):")
print(json.dumps(tools_definition[0], indent=2, ensure_ascii=False))

LLM ma teraz 3 narzędzia do wyboru:
  get_weather: Sprawdza aktualną pogodę w podanym mieście (temperatura, opi...
  calculate: Wykonuje obliczenie matematyczne. Użyj gdy trzeba coś policz...
  search_presidents: Przeszukuje bazę prezydentów Polski (III RP). Pola: kadencja...

Przykładowa definicja (get_weather):
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Sprawdza aktualną pogodę w podanym mieście (temperatura, opis, wilgotność, godzina pomiaru).",
    "parameters": {
      "properties": {
        "city": {
          "description": "Nazwa miasta, np. 'Kraków', 'Warszawa'",
          "title": "City",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "type": "object"
    }
  }
}


In [15]:
# ── Model ToolReasoning ──────────────────────────────────────────────
# Modele mogą mieć wbudowany tok myślenia:
#   - Qwen3 → atrybut reasoning_content
#   - Gemma-4 → tokeny <|channel>thought w msg.content
# Jeśli model NIE ma natywnego reasoning — wymuszamy go przez instructor.
#
# ⚠️ UWAGA: Wymuszony reasoning = DWA OSOBNE zapytania — jedno o reasoning,
# drugie o właściwy tool call. Model może teoretycznie podjąć RÓŻNE decyzje!

class ToolReasoning(BaseModel):
    thinking: str = Field(..., description="Krótkie uzasadnienie — dlaczego wybrałeś to narzędzie (1-2 zdania)")
    needs_tool: bool = Field(..., description="Czy pytanie wymaga użycia narzędzia?")
    tool_name: Optional[str] = Field(None, description="Nazwa narzędzia do użycia (lub None)")
    confidence: float = Field(..., ge=0, le=1, description="Pewność wyboru w skali 0-1")


def ask_with_tools(question, verbose=True, show_reasoning=True):
    if not client:
        print("LLM niedostępny.")
        return None

    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    response = client.chat.completions.create(
        model=MODEL_NAME, messages=messages, tools=tools_definition, temperature=0.1
    )

    assistant_msg = response.choices[0].message

    # ── Reasoning: natywny (Qwen3, Gemma-4) → fallback na instructor ──
    if verbose and show_reasoning:
        native_reasoning = extract_reasoning(assistant_msg)
        if native_reasoning:
            print_reasoning(assistant_msg)
        elif instructor_client:
            try:
                forced = instructor_client.chat.completions.create(
                    model=MODEL_NAME,
                    response_model=ToolReasoning,
                    messages=[
                        {"role": "system", "content":
                         f"Masz dostępne narzędzia: {list(AVAILABLE_TOOLS.keys())}. "
                         "Zdecyduj które narzędzie użyć (lub żadne). "
                         "Odpowiedz PŁASKIM JSON-em z polami: thinking, needs_tool, tool_name, confidence."},
                        {"role": "user", "content": question}
                    ],
                )
                print(f"  🔍 Reasoning (wymuszony przez instructor):")
                print(f"     Myślenie:   {forced.thinking}")
                print(f"     Narzędzie:  {forced.tool_name or 'BRAK'} (pewność: {forced.confidence:.0%})")
                print()
            except Exception:
                pass

    content = clean_content(assistant_msg)
    if verbose and content and assistant_msg.tool_calls:
        print(f"  💬 LLM mówi: {content[:300]}")
        print()

    if not assistant_msg.tool_calls:
        if verbose:
            print(f"  Narzędzie: BRAK (LLM odpowiedział sam)")
            print()
            display(Markdown(content or ""))
        return content

    messages.append(assistant_msg)
    n_calls = len(assistant_msg.tool_calls)
    for i, tool_call in enumerate(assistant_msg.tool_calls, 1):
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)

        label = f"[{i}/{n_calls}] " if n_calls > 1 else ""
        if verbose:
            print(f"  {label}Narzędzie: {func_name}({func_args})")

        result = AVAILABLE_TOOLS.get(func_name, lambda **kw: "Nieznane narzędzie")(**func_args)
        if verbose:
            print(f"  {label}Wynik:     {result[:150]}")
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    final = client.chat.completions.create(model=MODEL_NAME, messages=messages, temperature=0.1)
    final_answer = clean_content(final.choices[0].message)

    if verbose:
        print()
        display(Markdown(final_answer or ""))
    return final_answer

print("Funkcja ask_with_tools() gotowa!")

Funkcja ask_with_tools() gotowa!


In [16]:
pytania = [
    "Jaka jest pogoda w Gdańsku?",
    "Ile to jest 17 razy 23?",
    "Kto był najmłodszym prezydentem Polski?",
    "Co to jest Python?",
    "Oblicz pierwiastek z 256 i powiedz mi pogodę w Poznaniu",
]

if client:
    for q in pytania:
        print(f"\n{'═'*60}")
        print(f"PYTANIE: {q}")
        print(f"{'═'*60}")
        ask_with_tools(q)
else:
    print("LLM niedostępny.")


════════════════════════════════════════════════════════════
PYTANIE: Jaka jest pogoda w Gdańsku?
════════════════════════════════════════════════════════════


  🧠 Tok myślenia (reasoning):
     1. **Analyze the user's request:** The user is asking "Jaka jest pogoda w Gdańsku?" (What is the weather in Gdańsk?).
     
     2. **Examine available tools:** I have three tools: `get_weather`, `calculate`, and `search_presidents`.
     
     3. **Determine the relevant tool:** The request is about weather, so `get_weather` is the appropriate tool.
     
     4. **Identify required arguments for the chosen tool:** The `get_weather` function requires a `city` argument (a string).
     
     5. **Extract arguments from the user's requ

  Narzędzie: get_weather({'city': 'Gdańsk'})
  Wynik:     Gdańsk: 12°C, Słonecznie, wilgotność 68%, dane z 2026-05-10 07:58 PM (wttr.in)


W Gdańsku jest **12°C**, a pogoda jest **słoneczna**. Wilgotność wynosi 68%. (Dane z 10 maja 2026, godz. 19:58).


════════════════════════════════════════════════════════════
PYTANIE: Ile to jest 17 razy 23?
════════════════════════════════════════════════════════════


  🧠 Tok myślenia (reasoning):
     The user is asking for the result of a multiplication: 17 times 23.
     I have a tool `calculate` that can perform mathematical calculations.
     I should use this tool with the expression "17*23".
     
     1. **Identify the intent:** The user wants a calculation performed.
     2. **Select the tool:** `calculate`.
     3. **Determine parameters:** The expression is "17*23".
     4. **Construct the tool call.**

  Narzędzie: calculate({'expression': '17*23'})
  Wynik:     17*23 = 391


...

  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking "Co to jest Python?" (What is Python?).
     
     2. **Examine Available Tools:**
         * `get_weather`: For weather information. Irrelevant.
         * `calculate`: For mathematical calculations. Irrelevant.
         * `search_presidents`: For information about Polish presidents. Irrelevant.
     
     3. **Determine Tool Applicability:** None of the provided tools are capable of answering a general knowledge question about programming languages like Python.
     
     4. **Formulate the 

  Narzędzie: BRAK (LLM odpowiedział sam)



Python to bardzo popularny, wysokopoziomowy język programowania. Jest ceniony za swoją **czytelność** i prosty, intuicyjny składnia, która przypomina naturalny język angielski. Dzięki temu jest często polecany dla początkujących programistów, ale jednocześnie jest potężnym narzędziem używanym przez profesjonalistów.

Oto kluczowe cechy i zastosowania Pythona:

### Kluczowe Cechy
1. **Czytelność (Readability):** Python kładzie duży nacisk na czytelny kod. Jego składnia jest minimalistyczna i wymaga mniej linii kodu niż wiele innych języków, aby wykonać to samo zadanie.
2. **Interpretowany:** Kod Pythona jest wykonywany przez interpreter, co ułatwia proces testowania i debugowania.
3. **Wieloparadygmatyczny:** Obsługuje różne style programowania (obiektowy, proceduralny, funkcyjny).
4. **Ogólnego przeznaczenia (General-Purpose):** Nie jest ograniczony do jednej dziedziny – można go używać niemal wszędzie.

### Główne Zastosowania
Python jest niezwykle wszechstronny i znajduje zastosowanie w wielu dziedzinach:

* **Data Science i Analiza Danych:** Jest to jeden z najsilniejszych punktów Pythona. Biblioteki takie jak **Pandas**, **NumPy** i **Matplotlib** są standardem w tej dziedzinie.
* **Sztuczna Inteligencja i Uczenie Maszynowe (AI/ML):** Frameworki takie jak **TensorFlow**, **PyTorch** i **Scikit-learn** są zbudowane w dużej mierze na bazie Pythona.
* **Rozwój Backendu Webowego:** Do tworzenia serwerów i logiki aplikacji internetowych używa się frameworków takich jak **Django** i **Flask**.
* **Automatyzacja i Skrypty:** Jest idealny do pisania małych skryptów automatyzujących powtarzalne zadania (np. zarządzanie plikami, scraping stron internetowych).
* **Rozwój Oprogramowania:** Stosowany również w tworzeniu narzędzi desktopowych.

**Podsumowując:** Jeśli chcesz szybko i czytelnie rozwiązać problem, a jednocześnie wejść w świat nowoczesnych technologii (AI, Big Data), Python jest jednym z najlepszych punktów startowych.


════════════════════════════════════════════════════════════
PYTANIE: Oblicz pierwiastek z 256 i powiedz mi pogodę w Poznaniu
════════════════════════════════════════════════════════════


  🧠 Tok myślenia (reasoning):
     The user has two distinct requests:
     1.  Calculate the square root of 256 ($\sqrt{256}$).
     2.  Get the weather in Poznań.
     
     I have two tools that can handle these requests:
     1.  `calculate`: For mathematical operations like finding the square root.
     2.  `get_weather`: For checking the weather in a specific city.
     
     I should call both tools sequentially or in parallel if possible, but since the model output format requires a sequence of tool calls, I will generate both necessary calls.
     
     **Step 1: Calcul

  [1/2] Narzędzie: calculate({'expression': 'sqrt(256)'})
  [1/2] Wynik:     sqrt(256) = 16.0
  [2/2] Narzędzie: get_weather({'city': 'Poznań'})
  [2/2] Wynik:     Poznań: 19°C, Słonecznie, wilgotność 35%, dane z 2026-05-10 07:46 PM (wttr.in)


Pierwiastek z 256 wynosi **16**.

Pogoda w Poznaniu to: **19°C, Słonecznie**, wilgotność 35% (dane z 2026-05-10 07:46 PM).

### Ćwiczenie 1: Co ważniejsze — nazwa czy opis narzędzia?

LLM wybiera narzędzie na podstawie **nazwy** (`name`) i **opisu** (`description`).
Ale co ma większy wpływ na jego decyzję? Sprawdźmy!

**Część A — Zadanie: zmień opis i sprawdź efekt**

Zmień `description` narzędzia `get_weather` na coś kompletnie mylącego.
Zadaj pytanie o pogodę i obserwuj: czy LLM się pomylił?

In [17]:
# Ćwiczenie 1A: Zmień description get_weather na coś mylącego
import copy

# Kopia — nie modyfikujemy oryginału!
test_tools = copy.deepcopy(tools_definition)

# --- TUTAJ ZMIEŃ description ---

test_tools[0]["function"]["description"] = "Przeszukuje bazę danych o prezydentach Polski"

if test_tools[0]["function"]["description"] is ...:
    print("\033[1;31m⬆️ Uzupełnij description powyżej!\033[0m")

###### <span style="color: #c17f24;">Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Wpisz np.: `test_tools[0]["function"]["description"] = "Przeszukuje bazę danych o prezydentach Polski"`

###### <span style="color: #5a8a6a;">Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [18]:
# Ćwiczenie 1A — rozwiązanie
import copy

test_tools = copy.deepcopy(tools_definition)
test_tools[0]["function"]["description"] = "Przeszukuje bazę danych o prezydentach Polski"

##### 

In [19]:
# --- Test ćwiczenia 1A (uruchom po uzupełnieniu komórki powyżej) ---
def test_1a(pytanie):
    print(f"Nowy opis get_weather: '{test_tools[0]['function']['description']}'")
    print(f"Ale nazwa to nadal:   '{test_tools[0]['function']['name']}'")
    print(f"Pytanie: '{pytanie}'")
    print()
    if client:
        messages = [
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user", "content": pytanie}
        ]
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME, messages=messages, tools=test_tools, temperature=0.1
            )
            msg = response.choices[0].message
            print_reasoning(msg)
            if msg.tool_calls:
                tc = msg.tool_calls[0]
                print(f"  LLM wybrał: {tc.function.name}({tc.function.arguments})")
            else:
                print(f"  LLM odpowiedział bez narzędzia: {(clean_content(msg) or '')[:200]}")
        except Exception as e:
            print(f"\033[1;33m⚠️ Model zwrócił błąd: {e}\033[0m")
            print("Spróbuj uruchomić komórkę ponownie — niektóre modele czasem się zawieszają.")

# ═══ TESTUJ TUTAJ — zmień pytanie i uruchom komórkę ponownie! ═══
test_1a("Jaka jest pogoda we Wrocławiu?")

Nowy opis get_weather: 'Przeszukuje bazę danych o prezydentach Polski'
Ale nazwa to nadal:   'get_weather'
Pytanie: 'Jaka jest pogoda we Wrocławiu?'



  🧠 Tok myślenia (reasoning):
     1. **Analyze the user's request:** The user is asking "Jaka jest pogoda we Wrocławiu?" (What is the weather in Wrocław?).
     
     2. **Examine available tools:**
         * `get_weather`: This tool retrieves weather information for a specified city. It requires a `city` parameter.
         * `calculate`: This tool performs mathematical calculations. Not relevant here.
         * `search_presidents`: This tool searches a database of Polish presidents. Not relevant here.
     
     3. **Determine the appropriate tool:** The `get

  LLM wybrał: get_weather({"city":"Wrocław"})


###### 

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:14px; border-radius:4px;">

**Co zaobserwowałeś?**

Większość modeli **i tak wywoła `get_weather`** — bo nazwa `"get_weather"` jest zbyt silnym sygnałem!
LLM czyta **nazwę funkcji** i na jej podstawie podejmuje decyzję, często ignorując opis.

**Wniosek z części A:** Sama zmiana opisu może nie wystarczyć. Nazwa funkcji to bardzo silna wskazówka.

</div>

### Ćwiczenie 1 — Część B: Cichy błąd (najgroźniejszy rodzaj!)

Skoro LLM opiera się na nazwie — ukryjmy ją! Użyjemy **neutralnych nazw** (`tool_alpha`, `tool_beta`)
i **zamienimy opisy** dwóch funkcji, które przyjmują **ten sam typ argumentu** (`city: str`).

Kluczowy trik: obie funkcje akceptują `"Kraków"` i obie zwracają sensowny wynik —
ale **w zupełnie innym kontekście**. Nie będzie żadnego błędu, żadnego crashu.
Będą po prostu **złe dane**.

In [20]:
# Ćwiczenie 1B (demo): Cichy błąd — dwie funkcje z tym samym parametrem + zamienione opisy

# Pomocnicza funkcja — informacje o mieście (populacja, zabytki)
def city_info(city: str) -> str:
    """Zwraca informacje o polskim mieście — populacja, zabytki, historia."""
    dane = {
        "Kraków": "Kraków: ~800 tys. mieszkańców, Wawel, Sukiennice, Uniwersytet Jagielloński, stolica Małopolski.",
        "Warszawa": "Warszawa: ~1.86 mln mieszkańców, stolica Polski, Pałac Kultury i Nauki, Stare Miasto.",
        "Gdańsk": "Gdańsk: ~470 tys. mieszkańców, Żuraw, Stocznia Gdańska, Westerplatte, Długi Targ.",
        "Poznań": "Poznań: ~535 tys. mieszkańców, Stary Rynek, Koziołki Poznańskie, Międzynarodowe Targi.",
        "Wrocław": "Wrocław: ~674 tys. mieszkańców, Ostrów Tumski, krasnale, Hala Stulecia.",
    }
    return dane.get(city, f"Brak informacji o mieście: {city}")

# Tworzymy DWA narzędzia z neutralnymi nazwami i TYM SAMYM parametrem (city)
class CityArg(BaseModel):
    city: str = Field(..., description="Nazwa miasta, np. 'Kraków'")

weather_desc = "Sprawdza aktualną pogodę w polskim mieście — temperaturę i warunki atmosferyczne."
city_desc    = "Zwraca informacje o polskim mieście — populacja, zabytki, historia."

# Tworzymy narzędzia z NEUTRALNYMI nazwami
tool_alpha = make_tool("tool_alpha", weather_desc, CityArg)
tool_beta  = make_tool("tool_beta",  city_desc,    CityArg)

# 🔀 SABOTAŻ: zamieniamy opisy!
tool_alpha["function"]["description"] = city_desc     # tool_alpha WYGLĄDA jak info o mieście
tool_beta["function"]["description"]  = weather_desc  # tool_beta  WYGLĄDA jak pogoda

# Ale mapowanie do PRAWDZIWYCH funkcji zostaje oryginalne:
sabotaged_dispatch = {
    "tool_alpha": get_weather,  # ma opis "info o mieście" ale NAPRAWDĘ sprawdza pogodę!
    "tool_beta":  city_info,    # ma opis "pogoda" ale NAPRAWDĘ zwraca info o mieście!
}

sabotaged_tools = [tool_alpha, tool_beta]

print("Sabotaż gotowy! Oto co LLM zobaczy:\n")
for t in sabotaged_tools:
    n = t["function"]["name"]
    d = t["function"]["description"]
    real_fn = sabotaged_dispatch[n]
    print(f"  {n}:")
    print(f"    Opis:             \"{d}\"")
    print(f"    Prawdziwa f-ja:   {real_fn.__name__}()")
    print()

Sabotaż gotowy! Oto co LLM zobaczy:

  tool_alpha:
    Opis:             "Zwraca informacje o polskim mieście — populacja, zabytki, historia."
    Prawdziwa f-ja:   get_weather()

  tool_beta:
    Opis:             "Sprawdza aktualną pogodę w polskim mieście — temperaturę i warunki atmosferyczne."
    Prawdziwa f-ja:   city_info()



In [21]:
# Ćwiczenie 1B (test): Wysyłamy pytanie — LLM widzi podmienione opisy

if client:
    # Nazwy narzędzi w sabotażu to tool_alpha/tool_beta — podajemy je do reasoning
    sabotaged_names = [t["function"]["name"] for t in sabotaged_tools]

    for question in ["Jaka jest pogoda we Wrocławiu?", "Opowiedz mi o Wrocławiu jako mieście"]:
        print(f"{'═'*60}")
        print(f"Pytanie: \"{question}\"\n")

        # Wymuszony reasoning — zobaczmy CO model myśli o sabotażowych narzędziach
        if instructor_client:
            try:
                reasoning = instructor_client.chat.completions.create(
                    model=MODEL_NAME,
                    response_model=ToolReasoning,
                    messages=[
                        {"role": "system", "content":
                         f"Masz dostępne narzędzia: {sabotaged_names}. "
                         f"Opisy: {[t['function']['description'] for t in sabotaged_tools]}. Zdecyduj. "
                         "Odpowiedz PŁASKIM JSON-em z polami: thinking, needs_tool, tool_name, confidence."},
                        {"role": "user", "content": question}
                    ],
                )
                print(f"  🔍 Reasoning (wymuszony):")
                print(f"     Myślenie:   {reasoning.thinking}")
                print(f"     Narzędzie:  {reasoning.tool_name or 'BRAK'} (pewność: {reasoning.confidence:.0%})")
                print()
            except Exception:
                pass

        messages = [
            {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]

        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=sabotaged_tools, temperature=0.1
        )
        msg = response.choices[0].message

        print_reasoning(msg)

        if msg.tool_calls:
            tc = msg.tool_calls[0]
            chosen_name = tc.function.name
            chosen_args = json.loads(tc.function.arguments)
            real_func = sabotaged_dispatch[chosen_name]

            print(f"  LLM wybrał:        {chosen_name}({chosen_args})")
            print(f"  Prawdziwa funkcja:  {real_func.__name__}()")

            result = real_func(**chosen_args)
            print(f"  Wynik:             \"{result}\"")
            print()

            # Analiza
            if "pogoda" in question.lower():
                if real_func.__name__ == "get_weather":
                    print("  ✅ Poprawnie! Dostaliśmy pogodę.")
                else:
                    print("  🤦 CICHY BŁĄD! Pytaliśmy o pogodę, a dostaliśmy info o mieście.")
                    print("     Żadnego errora — funkcja się wykonała, wynik wygląda sensownie.")
                    print("     Ale to NIE jest pogoda!")
            else:
                if real_func.__name__ == "city_info":
                    print("  ✅ Poprawnie! Dostaliśmy info o mieście.")
                else:
                    print("  🤦 CICHY BŁĄD! Pytaliśmy o miasto, a dostaliśmy pogodę.")
                    print("     Żadnego errora — ale odpowiedź jest kompletnie nie na temat!")
        else:
            print(f"  LLM odpowiedział bez narzędzia: {(clean_content(msg) or '')[:200]}")
        print()
else:
    print("LLM niedostępny.")

════════════════════════════════════════════════════════════
Pytanie: "Jaka jest pogoda we Wrocławiu?"



  🔍 Reasoning (wymuszony):
     Myślenie:   Użytkownik pyta o aktualną pogodę we Wrocławiu, a narzędzie 'tool_beta' jest przeznaczone do sprawdzania pogody w polskich miastach.
     Narzędzie:  tool_beta (pewność: 95%)



  🧠 Tok myślenia (reasoning):
     The user is asking for the weather in Wrocław. I have a tool `tool_beta` that can check the current weather in a Polish city. I should call this tool with "Wrocław" as the `city` argument.

  LLM wybrał:        tool_beta({'city': 'Wrocław'})
  Prawdziwa funkcja:  city_info()
  Wynik:             "Wrocław: ~674 tys. mieszkańców, Ostrów Tumski, krasnale, Hala Stulecia."

  🤦 CICHY BŁĄD! Pytaliśmy o pogodę, a dostaliśmy info o mieście.
     Żadnego errora — funkcja się wykonała, wynik wygląda sensownie.
     Ale to NIE jest pogoda!

════════════════════════════════════════════════════════════
Pytanie: "Opowiedz mi o Wrocławiu jako mieście"



  🔍 Reasoning (wymuszony):
     Myślenie:   Użytkownik prosi o informacje na temat Wrocławia jako miasta, co jest dokładnie tym, co oferuje narzędzie tool_alpha.
     Narzędzie:  tool_alpha (pewność: 95%)



  🧠 Tok myślenia (reasoning):
     The user is asking for information about the city of Wrocław. I have a tool `tool_alpha` that can provide information about Polish cities, including population, monuments, and history. This tool is suitable for answering the user's request. I should call `tool_alpha` with "Wrocław" as the city argument.

  LLM wybrał:        tool_alpha({'city': 'Wrocław'})
  Prawdziwa funkcja:  get_weather()
  Wynik:             "Wrocław: 19°C, Słonecznie, wilgotność 43%, dane z 2026-05-10 07:52 PM (wttr.in)"

  🤦 CICHY BŁĄD! Pytaliśmy o miasto, a dostaliśmy pogodę.
     Żadnego errora — ale odpowiedź jest kompletnie nie na temat!



<div style="background:#f8d7da; border-left:4px solid #dc3545; padding:14px; border-radius:4px;">

**Cichy błąd jest GORSZY od crashu!**

Gdyby funkcja się wysypała — zobaczylibyśmy `Error` i wiedzielibyśmy, że coś nie gra.
Ale tutaj **wszystko wygląda OK**: funkcja się wykonała, wynik jest poprawny gramatycznie,
nie ma żadnego ostrzeżenia. Jedyny problem? **Dane są kompletnie nie te.**

W produkcyjnych systemach to prowadzi do:
- Chatbot podaje złe informacje z przekonaniem
- Agent wykonuje niewłaściwą akcję (np. kasuje plik zamiast go wyszukać)
- Użytkownik nie wie, że odpowiedź jest błędna

**Dlatego nazwy i opisy narzędzi to krytyczny element bezpieczeństwa AI systemów!**

</div>

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; border-radius:4px; margin-top:10px;">

**Wnioski z Ćwiczenia 1:**

| Źródło błędu | Efekt | Wykrywalność |
|---|---|---|
| Zły opis narzędzia (ale znana nazwa) | LLM i tak trafia | Żaden — nam się udało! |
| Podmienione nazwy + opisy | **Cichy błąd** — złe narzędzie, dane nie na temat | Trudna — trzeba sprawdzić |

Dlatego w produkcyjnych systemach AI kluczowe są:
**dobre opisy narzędzi** + **testy jakości odpowiedzi**

</div>

###### 

### Ćwiczenie 2: Dodaj własne narzędzie

Stwórz nową funkcję i dodaj jej opis do `tools_definition`.

**Zadanie:** Napisz narzędzie `get_population(city)` które zwraca przybliżoną liczbę
mieszkańców polskiego miasta.

**Dane do użycia** (GUS 2024):

| Miasto | Mieszkańcy | Ranking |
|---|---|---|
| Warszawa | ~1 860 000 | #1 |
| Kraków | ~800 000 | #2 |
| Wrocław | ~674 000 | #3 |
| Łódź | ~646 000 | #4 |
| Poznań | ~535 000 | #5 |
| Gdańsk | ~470 000 | #6 |

**Jak to zrobić:**
1. Przechowaj dane w **słowniku** `dict` — klucz to nazwa miasta, wartość to **tupla** `(populacja, ranking)`
2. Funkcja zwraca **f-string** z informacjami, np. `"Kraków: ~800,000 mieszkańców (#2 w Polsce)"`
3. Dodaj do `AVAILABLE_TOOLS` i do `tools_definition` (przez `make_tool`)

In [22]:
# Ćwiczenie 2: Dodaj narzędzie get_population

# Krok 1: Zdefiniuj funkcję

def get_population(city: str) -> str:
    dane = {
        "Warszawa": (1_860_000, 1), "Kraków": (800_000, 2), "Wrocław": (674_000, 3),
        "Gdańsk": (470_000, 6), "Poznań": (535_000, 5), "Łódź": (646_000, 4),
    }
    if city in dane:
        pop, rank = dane[city]
        return f"{city}: ~{pop:,} mieszkańców (#{rank} w Polsce)"
    return f"Brak danych o populacji dla: {city}"

# Krok 2: Dodaj do AVAILABLE_TOOLS (nie zmieniaj)
AVAILABLE_TOOLS["get_population"] = get_population

# Krok 3: Zdefiniuj model argumentów i dodaj do tools_definition

class GetPopulationArgs(BaseModel):
    city: str = Field(..., description="Nazwa miasta, np. 'Kraków'")

_population_desc = "Zwraca przybliżoną liczbę mieszkańców polskiego miasta."

# Usuwamy stary wpis (bezpieczne przy wielokrotnym uruchamianiu)
tools_definition = [t for t in tools_definition if t["function"]["name"] != "get_population"]

if _population_desc is ...:
    print("\033[1;33m⬆️ Uzupełnij opis narzędzia powyżej!\033[0m")
else:
    tools_definition.append(
        make_tool("get_population", _population_desc, GetPopulationArgs)
    )

# --- TEST (nie zmieniaj) ---
_test_result = get_population("Wrocław")
if _test_result is None:
    print("\033[1;31m⬆️ Uzupełnij funkcję get_population! Teraz zwraca None (pass).\033[0m")
elif _population_desc is ...:
    pass  # komunikat już wyświetlony powyżej
elif "get_population" not in AVAILABLE_TOOLS:
    print("\033[1;31m⬆️ Dodaj get_population do AVAILABLE_TOOLS!\033[0m")
else:
    print(f"Mamy {len(AVAILABLE_TOOLS)} narzędzi: {list(AVAILABLE_TOOLS.keys())}")
    print(f"\nTest bezpośredni: {get_population('Wrocław')}")
    if client:
        print()
        ask_with_tools("Ile mieszkańców ma Wrocław?")

Mamy 4 narzędzi: ['get_weather', 'calculate', 'search_presidents', 'get_population']

Test bezpośredni: Wrocław: ~674,000 mieszkańców (#3 w Polsce)



  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking "Ile mieszkańców ma Wrocław?" (How many inhabitants does Wrocław have?).
     
     2. **Examine Available Tools:** I have four tools:
         * `get_weather`: Gets weather information. (Not relevant)
         * `calculate`: Performs mathematical calculations. (Not relevant)
         * `search_presidents`: Searches for information about Polish presidents. (Not relevant)
         * `get_population`: Returns the approximate population of a Polish city. (Relevant)
     
     3. **Determine 

  Narzędzie: get_population({'city': 'Wrocław'})
  Wynik:     Wrocław: ~674,000 mieszkańców (#3 w Polsce)


Odpowiedź: Wrocław ma około **674 000 mieszkańców**. Jest to trzecie co do wielkości miasto w Polsce.

###### <span style="color: #c17f24;">Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Wzoruj się na `get_weather` — słownik z danymi, f-string dla wyniku. Do `tools_definition` użyj `make_tool()` — potrzebujesz model Pydantic dla **argumentów** (tu: `city: str`) + `tools_definition.append(make_tool("get_population", "...", GetPopulationArgs))`.

###### <span style="color: #5a8a6a;">Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [23]:
# Ćwiczenie 2 — rozwiązanie

def get_population(city: str) -> str:
    dane = {
        "Warszawa": (1_860_000, 1), "Kraków": (800_000, 2), "Wrocław": (674_000, 3),
        "Gdańsk": (470_000, 6), "Poznań": (535_000, 5), "Łódź": (646_000, 4),
    }
    if city in dane:
        pop, rank = dane[city]
        return f"{city}: ~{pop:,} mieszkańców (#{rank} w Polsce)"
    return f"Brak danych o populacji dla: {city}"

AVAILABLE_TOOLS["get_population"] = get_population

class GetPopulationArgs(BaseModel):
    city: str = Field(..., description="Nazwa miasta, np. 'Kraków'")

# Usuwamy stary wpis (np. z nieuzupełnionej komórki powyżej)
tools_definition = [t for t in tools_definition if t["function"]["name"] != "get_population"]
tools_definition.append(
    make_tool("get_population", "Zwraca przybliżoną liczbę mieszkańców polskiego miasta.", GetPopulationArgs)
)

# --- TEST ---
print(f"Mamy {len(AVAILABLE_TOOLS)} narzędzi: {list(AVAILABLE_TOOLS.keys())}")
print(f"\nTest bezpośredni: {get_population('Wrocław')}")
if client:
    print()
    ask_with_tools("Ile mieszkańców ma Wrocław?")

Mamy 4 narzędzi: ['get_weather', 'calculate', 'search_presidents', 'get_population']

Test bezpośredni: Wrocław: ~674,000 mieszkańców (#3 w Polsce)



  🧠 Tok myślenia (reasoning):
     1. **Analyze the user's request:** The user is asking "Ile mieszkańców ma Wrocław?" (How many inhabitants does Wrocław have?).
     
     2. **Examine available tools:** I have four tools:
         * `get_weather`: Gets weather information. (Irrelevant)
         * `calculate`: Performs mathematical calculations. (Irrelevant)
         * `search_presidents`: Searches a database of Polish presidents. (Irrelevant)
         * `get_population`: Returns the approximate population of a Polish city. (Relevant)
     
     3. **Determine the nec

  Narzędzie: get_population({'city': 'Wrocław'})
  Wynik:     Wrocław: ~674,000 mieszkańców (#3 w Polsce)


Odpowiedź: Wrocław ma około **674 000 mieszkańców**. Jest to trzecie co do wielkości miasto w Polsce.

###### 

## 7. Wikipedia jako narzędzie

Pora na **prawdziwe** narzędzie sięgające do internetu!
Biblioteka `wikipedia` pozwala przeszukiwać Wikipedię programatycznie.

### Ćwiczenie 3: Zbuduj narzędzie Wikipedia

**Zadanie:**
1. Napisz funkcję `search_wikipedia(query)` która zwraca tytuł, streszczenie i URL artykułu jako tekst
2. Dodaj definicję narzędzia do `tools_definition` + do `AVAILABLE_TOOLS`
3. Przetestuj pytaniem do LLM-a

In [24]:
import wikipedia
wikipedia.set_lang("pl")

# Biblioteka wikipedia:
#   wikipedia.search("Wrocław")     → lista tytułów: ["Wrocław", "Wrocław (ujednoznacznienie)", ...]
#   wikipedia.page("Wrocław")       → obiekt strony z polami:
#       page.title   → "Wrocław"
#       page.summary → "Wrocław – miasto na prawach powiatu..."  (pełne streszczenie)
#       page.url     → "https://pl.wikipedia.org/wiki/Wrocław"
#
# Uwaga: wikipedia.page() może rzucić DisambiguationError — złap go w try/except.

def search_wikipedia(query: str) -> str:
    """Przeszukuje Wikipedię i zwraca tytuł + streszczenie artykułu."""
    try:
        results = wikipedia.search(query, results=3)
        if not results:
            return f"Nie znaleziono artykułów dla: {query}"

        try:
            page = wikipedia.page(results[0])
        except wikipedia.DisambiguationError as e:
            page = wikipedia.page(e.options[0])

        return f"{page.title}\n\n{page.summary[:500]}\n\nŹródło: {page.url}"

    except Exception as e:
        return f"Błąd wyszukiwania Wikipedia: {e}"

# --- Rejestracja narzędzia ---
AVAILABLE_TOOLS["search_wikipedia"] = search_wikipedia

class SearchWikipediaArgs(BaseModel):

    query: str = Field(..., description="Zapytanie do Wikipedii, np. 'fotosynteza', 'Nikola Tesla'"
    )

_wikipedia_desc = "Przeszukuje Wikipedię i zwraca streszczenie artykułu. Użyj gdy pytanie dotyczy wiedzy ogólnej, historii, nauki, geografii."

# Usuwamy stary wpis (bezpieczne przy wielokrotnym uruchamianiu)
tools_definition = [t for t in tools_definition if t["function"]["name"] != "search_wikipedia"]

if _wikipedia_desc is ...:
    print("\033[1;33m⬆️ Uzupełnij opis narzędzia powyżej!\033[0m")
else:
    tools_definition.append(
        make_tool("search_wikipedia", _wikipedia_desc, SearchWikipediaArgs)
    )

# --- TEST (nie zmieniaj) ---
_test_result = search_wikipedia("Wrocław")
if _test_result is None or _test_result is ...:
    print("\033[1;31m⬆️ Uzupełnij return w search_wikipedia!\033[0m")
elif _wikipedia_desc is ...:
    pass  # komunikat już wyświetlony powyżej
elif "search_wikipedia" not in AVAILABLE_TOOLS:
    print("\033[1;31m⬆️ Dodaj search_wikipedia do AVAILABLE_TOOLS!\033[0m")
else:
    print("Test bezpośredni:")
    print(_test_result[:300])
    print(f"\nMamy {len(AVAILABLE_TOOLS)} narzędzi: {list(AVAILABLE_TOOLS.keys())}")
    if client:
        print("\nTest przez LLM:")
        ask_with_tools("Co to jest fotosynteza?")

Test bezpośredni:
Wrocław

Wrocław (łac. Vratislavia, niem. Breslau, dś. Brassel, Gruß-Brassel, cz. Vratislav) – miasto na prawach powiatu w południowo-zachodniej Polsce, siedziba władz województwa dolnośląskiego i powiatu wrocławskiego. Położone w Europie Środkowej, na Nizinie Śląskiej, nad Odrą i czterema jej dopły

Mamy 5 narzędzi: ['get_weather', 'calculate', 'search_presidents', 'get_population', 'search_wikipedia']

Test przez LLM:


  🧠 Tok myślenia (reasoning):
     1. **Analyze the user's request:** The user is asking "Co to jest fotosynteza?" (What is photosynthesis?).
     
     2. **Examine available tools:** I have several tools: `get_weather`, `calculate`, `search_presidents`, `get_population`, and `search_wikipedia`.
     
     3. **Determine the appropriate tool:** The question is about general knowledge, specifically a scientific concept (photosynthesis). The `search_wikipedia` tool is designed for "wiedzy ogólnej, historii, nauki, geografii" (general knowledge, histo

  Narzędzie: search_wikipedia({'query': 'fotosynteza'})


  Wynik:     Fotosynteza

Fotosynteza (stgr. φῶς – światło, σύνθεσις – łączenie) – proces wytwarzania związków organicznych z materii nieorganicznej zachodzący w k


Fotosynteza to **fundamentalny proces biochemiczny**, dzięki któremu rośliny, algi i niektóre bakterie (organizmy autotroficzne) przekształcają energię świetlną w energię chemiczną.

Mówiąc najprościej, jest to proces, w którym organizmy "tworzą jedzenie" (związki organiczne) z prostych, nieorganicznych składników, takich jak dwutlenek węgla i woda.

### 🌿 Jak to działa? (Składniki i produkty)

W procesie fotosyntezy organizmy wykorzystują:

1.  **Dwutlenek węgla ($\text{CO}_2$):** Pobierany z atmosfery.
2.  **Wodę ($\text{H}_2\text{O}$):** Pobieraną z gleby przez korzenie.
3.  **Światło słoneczne:** Stanowiące źródło energii.

A w wyniku tego procesu powstają:

1.  **Glukoza ($\text{C}_6\text{H}_{12}\text{O}_6$):** Cukier, który jest magazynowaną energią dla rośliny (jest to związek organiczny).
2.  **Tlen ($\text{O}_2$):** Uwalniany do atmosfery jako produkt uboczny.

### 🔬 Gdzie to się dzieje?

Fotosynteza zachodzi głównie w **chloroplastach** komórek roślinnych. Kluczowym elementem tego procesu jest **chlorofil** – zielony barwnik, który ma zdolność pochłaniania energii światła słonecznego.

### 🌍 Dlaczego fotosynteza jest tak ważna? (Znaczenie ekologiczne)

Fotosynteza ma absolutnie kluczowe znaczenie dla życia na Ziemi z dwóch głównych powodów:

1.  **Produkcja tlenu:** Jest to główne źródło tlenu w atmosferze, który jest niezbędny do oddychania większości organizmów żywych (w tym ludzi).
2.  **Podstawa łańcucha pokarmowego:** Rośliny, wytwarzając własne związki organiczne (cukry), stanowią podstawę niemal wszystkich ekosystemów lądowych i wodnych. Wszystkie zwierzęta, które zjadają rośliny (lub inne zwierzęta, które zjadają te rośliny), czerpią energię wytworzoną w procesie fotosyntezy.

---
**Podsumowując wzorem chemicznym:**

$$\text{Dwutlenek węgla} + \text{Woda} + \text{Energia świetlna} \rightarrow \text{Glukoza} + \text{Tlen}$$
$$\text{6CO}_2 + \text{6H}_2\text{O} + \text{Światło} \rightarrow \text{C}_6\text{H}_{12}\text{O}_6 + \text{6O}_2$$

###### <span style="color: #c17f24;">Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Funkcja: `wikipedia.search()` zwraca listę tytułów, `wikipedia.page()` zwraca stronę z polami `.title`, `.url`, `.summary`. Uważaj na `DisambiguationError` — złap go w `try/except`. Narzędzie zwraca zwykły tekst (f-string z tytułem, streszczeniem i URL). Do rejestracji: model Pydantic dla **argumentów** + `make_tool()`.

###### <span style="color: #5a8a6a;">Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [25]:
# Ćwiczenie 3 — rozwiązanie
import wikipedia
wikipedia.set_lang("pl")

def search_wikipedia(query: str) -> str:
    """Przeszukuje Wikipedię i zwraca tytuł + streszczenie artykułu."""
    try:
        results = wikipedia.search(query, results=3)
        if not results:
            return f"Nie znaleziono artykułów dla: {query}"

        try:
            page = wikipedia.page(results[0])
        except wikipedia.DisambiguationError as e:
            page = wikipedia.page(e.options[0])

        return f"{page.title}\n\n{page.summary[:500]}\n\nŹródło: {page.url}"

    except Exception as e:
        return f"Błąd wyszukiwania Wikipedia: {e}"

AVAILABLE_TOOLS["search_wikipedia"] = search_wikipedia

class SearchWikipediaArgs(BaseModel):
    query: str = Field(..., description="Zapytanie do Wikipedii, np. 'fotosynteza', 'Nikola Tesla'")

tools_definition = [t for t in tools_definition if t["function"]["name"] != "search_wikipedia"]
tools_definition.append(
    make_tool("search_wikipedia",
              "Przeszukuje Wikipedię i zwraca streszczenie artykułu. "
              "Użyj gdy pytanie dotyczy wiedzy ogólnej, historii, nauki, geografii.",
              SearchWikipediaArgs)
)

# --- TEST ---
print(f"Mamy {len(AVAILABLE_TOOLS)} narzędzi: {list(AVAILABLE_TOOLS.keys())}")
print(f"\nTest: {search_wikipedia('Wrocław')[:200]}...")
if client:
    print()
    ask_with_tools("Kim był Mikołaj Kopernik?")

Mamy 5 narzędzi: ['get_weather', 'calculate', 'search_presidents', 'get_population', 'search_wikipedia']



Test: Wrocław

Wrocław (łac. Vratislavia, niem. Breslau, dś. Brassel, Gruß-Brassel, cz. Vratislav) – miasto na prawach powiatu w południowo-zachodniej Polsce, siedziba władz województwa dolnośląskiego i pow...



  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking "Kim był Mikołaj Kopernik?" (Who was Nicolaus Copernicus?).
     
     2. **Identify the Intent:** The request is a general knowledge question about a historical figure (Nicolaus Copernicus).
     
     3. **Examine Available Tools:**
         * `get_weather`: For weather information (irrelevant).
         * `calculate`: For mathematical calculations (irrelevant).
         * `search_presidents`: For information about Polish presidents (irrelevant, Copernicus was not a president of th

  Narzędzie: search_wikipedia({'query': 'Mikołaj Kopernik'})


  Wynik:     Mikołaj Kopernik

Mikołaj Kopernik, łac. Nicolaus Copernicus, niem. Nikolaus Kopernikus (ur. 19 lutego 1473 w Toruniu, zm. w maju 1543 we Fromborku) –


Mikołaj Kopernik (łac. Nicolaus Copernicus) był wybitnym polskim polihistorem pochodzenia niemieckiego, który jest jednym z najważniejszych naukowców w historii ludzkości.

Oto kluczowe informacje na jego temat:

### 🔬 Naukowiec i Astronom
Kopernik jest przede wszystkim znany jako **astronom**, a jego najważniejsze osiągnięcie to opracowanie i promowanie **modelu heliocentrycznego**.

*   **Model Heliocentryczny:** Odrzucił on dotychczasowy, powszechnie akceptowany model geocentryczny (gdzie Ziemia jest nieruchomym centrum Wszechświata), proponując, że to **Słońce** znajduje się w centrum Układu Słonecznego, a Ziemia wraz z innymi planetami krąży wokół niego. Ta teoria zapoczątkowała rewolucję naukową w astronomii i fizyce.
*   **Dzieło:** Jego przełomowe dzieło, *De revolutionibus orbium coelestium* (O obrotach sfer niebieskich), opublikowane w 1543 roku, stanowi kamień milowy w historii nauki.

### 📚 Inne Role i Działalność
Kopernik był człowiekiem niezwykle wszechstronnym (polihistorem). Oprócz astronomii zajmował się również:

*   **Prawem i Administracją:** Był prawnikiem, urzędnikiem i dyplomatą.
*   **Medycyną:** Studiował medycynę.
*   **Inne dziedziny:** Jego zainteresowania obejmowały również matematykę, ekonomię, kartografię i filologię.

### 👤 Podsumowanie
Mikołaj Kopernik był uczonym, który nie tylko doskonale znał się na wielu dziedzinach, ale przede wszystkim zmienił fundamentalnie nasze rozumienie miejsca Ziemi we Wszechświecie, stając się ojcem nowoczesnej astronomii.

###### 

## 8. Web search — DuckDuckGo

Wikipedia jest świetna dla encyklopedycznej wiedzy. Ale co z **aktualnymi wydarzeniami**?

Biblioteka `ddgs` pozwala przeszukiwać internet przez DuckDuckGo — **bez klucza API**!

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; border-radius:4px;">
<b>Dlaczego DuckDuckGo a nie Google?</b> Google wymaga klucza API i ma płatne limity.
DuckDuckGo pozwala na darmowe wyszukiwanie programatyczne — idealne do prototypowania.
Uwaga: przy wielu zapytaniach może pojawić się rate limiting (tymczasowa blokada).
</div>

In [26]:
from ddgs import DDGS


def search_web(query: str) -> str:
    """
    Przeszukuje internet przez DuckDuckGo (pomija Wikipedię — od tego mamy osobne narzędzie).

    Args:
        query: Zapytanie, np. 'najnowsze wiadomości Polska'
    """
    try:
        # Pobieramy więcej wyników, bo część odfiltrujemy (Wikipedia)
        raw_results = DDGS().text(query, max_results=8)
        # Filtrujemy Wikipedię — mamy osobny tool search_wikipedia
        results = [r for r in raw_results if "wikipedia.org" not in r.get("href", "")][:3]
        if not results:
            return f"Brak wyników (poza Wikipedią) dla: {query}. Spróbuj search_wikipedia."
        parts = [f"Wyniki wyszukiwania: {query}\n"]
        for i, r in enumerate(results, 1):
            parts.append(f"{i}. {r['title']}\n   {r['body'][:200]}\n   {r['href']}")
        return "\n\n".join(parts)
    except Exception as e:
        return f"Wyszukiwanie niedostępne (DuckDuckGo): {e}. Spróbuj Wikipedii."


AVAILABLE_TOOLS["search_web"] = search_web

class SearchWebArgs(BaseModel):
    query: str = Field(..., description="Zapytanie do wyszukiwarki")

tools_definition = [t for t in tools_definition if t["function"]["name"] != "search_web"]
tools_definition.append(
    make_tool("search_web",
              "Przeszukuje internet przez DuckDuckGo. Użyj TYLKO gdy potrzebujesz aktualnych informacji, "
              "których nie ma na Wikipedii ani w bazie prezydentów.",
              SearchWebArgs)
)

print("Narzędzie search_web dodane!")
print(f"Mamy {len(AVAILABLE_TOOLS)} narzędzi: {list(AVAILABLE_TOOLS.keys())}")

print("\nTest bezpośredni:")
result = search_web("Python programming language")
print(result[:300] + "..." if len(result) > 300 else result)

Narzędzie search_web dodane!
Mamy 6 narzędzi: ['get_weather', 'calculate', 'search_presidents', 'get_population', 'search_wikipedia', 'search_web']

Test bezpośredni:


Wyniki wyszukiwania: Python programming language


1. Python (programming language)
   An overview of the Python programming language, including resources and methods for learning it.
   https://grokipedia.com/page/Python_programming_language

2. Welcome to Python.org
   Python is a versatile and ea...


Sprawdźmy, czy LLM wybierze `search_web` zamiast `search_wikipedia` — pytamy o coś **aktualnego i konkretnego**:

In [27]:
if client:
    # Pytanie o aktualne ceny → LLM powinien wybrać search_web (nie Wikipedię)
    ask_with_tools("Ile kosztuje bilet na pociąg Wrocław Warszawa?")

  🧠 Tok myślenia (reasoning):
     1. **Analyze the user's request:** The user is asking for the price of a train ticket from Wrocław to Warsaw ("Ile kosztuje bilet na pociąg Wrocław Warszawa?").
     
     2. **Examine available tools:**
         * `get_weather`: For weather information. (Irrelevant)
         * `calculate`: For mathematical calculations. (Irrelevant)
         * `search_presidents`: For information about Polish presidents. (Irrelevant)
         * `get_population`: For city population data. (Irrelevant)
         * `search_wikipedia`: For general kn

  Narzędzie: search_web({'query': 'cena biletu pociąg Wrocław Warszawa'})


  Wynik:     Wyniki wyszukiwania: cena biletu pociąg Wrocław Warszawa


1. Pociąg Kraków do Warszawa od 41 zł | Busbud
   Pociąg wyróżnia się jako dobry środek tra


Przepraszam, ale podane wyniki wyszukiwania nie zawierają aktualnej ceny biletu na trasie Wrocław-Warszawa. Ceny biletów kolejowych zmieniają się bardzo często w zależności od:

* **Daty podróży** (czy jest to weekend, święto, czy dzień powszedni)
* **Godziny odjazdu**
* **Rodzaju biletu** (normalny, promocyjny, itp.)
* **Przewoźnika** (PKP Intercity, itp.)

Aby uzyskać najdokładniejszą i aktualną cenę, **najlepiej jest sprawdzić bezpośrednio na stronach internetowych przewoźników kolejowych**, takich jak PKP Intercity, wpisując konkretne daty podróży.

Czy podasz mi preferowaną datę podróży, a spróbuję poszukać bardziej szczegółowych informacji?

## 8b. Tajemnica w danych — zagrożenie i szansa

Zanim przejdziemy do weryfikacji faktów, muszę wam coś powiedzieć.
W pliku `prezydenci_polski.md` **celowo ukryłem dwa zmyślone fakty** —
"mało znane" informacje, które brzmią wiarygodnie, ale są **całkowicie fikcyjne**.

Sprawdźmy, czy nasz agent je znajdzie i poda z pełnym przekonaniem:

In [28]:
if client:
    print("Test: Mało znane fakty o prezydentach")
    print("═"*60)
    ask_with_tools("Jakie mało znane fakty kryją polscy prezydenci? Sprawdź w bazie prezydentów, szukaj 'mało znany fakt'.")

Test: Mało znane fakty o prezydentach
════════════════════════════════════════════════════════════


  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking for "little-known facts" about Polish presidents and specifically instructs to use the "presidents database" by searching for "'mało znany fakt'".
     
     2. **Examine Available Tools:** I have several tools: `get_weather`, `calculate`, `search_presidents`, `get_population`, `search_wikipedia`, and `search_web`.
     
     3. **Select the Appropriate Tool:** The request explicitly mentions "polscy prezydenci" (Polish presidents) and asks to search a specific databas

  Narzędzie: search_presidents({'query': 'mało znany fakt'})
  Wynik:     ### Aleksander Kwaśniewski
  - kadencja: 1995–2005
  - lata życia: ur. 1954
  - miejsce urodzenia: Białogard
  - wiek w dniu objęcia urzędu: 41 lat
  


Polscy prezydenci mają wiele fascynujących historii, a baza danych dostarczyła kilku mało znanych faktów.

Oto dwa przykłady:

**1. Aleksander Kwaśniewski:**
* **Mało znany fakt:** W swoim gabinecie prezydenckim trzymał miniaturowe popiersie Kleopatry — podarunek od ambasadora Egiptu z 1997 roku, które stało się nieoficjalnym „talizmanem" Pałacu Prezydenckiego na czas jego kadencji.

**2. Andrzej Duda:**
* **Mało znany fakt:** Jest zapalonym kolekcjonerem antycznych monet — posiada jedną z największych prywatnych kolekcji monet z czasów Juliusza Cezara w Europie Środkowej, którą zaczął budować jeszcze na studiach prawniczych w Krakowie.

<div style="background:#f8d7da; border-left:4px solid #dc3545; padding:14px; border-radius:4px;">

**Oba te "fakty" są całkowicie ZMYŚLONE!**

- Kwaśniewski **nie miał** żadnego popiersia Kleopatry od ambasadora Egiptu
- Duda **nie kolekcjonuje** antycznych monet z czasów Juliusza Cezara

A mimo to nasz agent zwrócił je z pełnym przekonaniem — bo **nasze narzędzie** mu je podało.
Tych informacji **nie ma** na Wikipedii ani w internecie — nie da się ich zweryfikować z zewnątrz.

</div>

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:14px; border-radius:4px; margin-top:12px;">

**Zagrożenie: *garbage in, garbage out***

| Źródło błędu | Efekt | Wykrywalność |
|---|---|---|
| Zły opis narzędzia | LLM i tak trafia | Żaden — nam się udało! |
| Podmienione nazwy + opisy | **Cichy błąd** — złe narzędzie | Trudna — trzeba sprawdzić |
| Fałszywe dane w źródle | **Cichy błąd** — prawidłowe narzędzie, ale kłamstwo w danych | Bardzo trudna! |

Agent jest tak dobry (lub tak zły) jak jego narzędzia i dane.

</div>

<div style="background:#d4edda; border-left:4px solid #28a745; padding:14px; border-radius:4px; margin-top:12px;">

**Szansa: wewnętrzne repozytorium wiedzy**

Ale odwróćmy perspektywę — ten sam mechanizm to ogromna **szansa**!

Wyobraźcie sobie, że w waszej firmie jest:
- 🔒 **Know-how** — innowacyjna procedura, której nie chcecie upubliczniać
- 📋 **Regulacje wewnętrzne** — polityki, procedury, wytyczne dostępne tylko dla pracowników
- 🧪 **Dane wrażliwe** — wyniki badań, analizy rynkowe, strategie

Nie chcecie tego publikować w internecie — ale chcecie, żeby **LLM mógł podejmować decyzje na ich podstawie**.

Rozwiązanie: **wewnętrzne, strzeżone repozytorium wiedzy** (baza danych, pliki, API),
do którego LLM ma dostęp **wyłącznie przez function calling**. Dokładnie tak, jak nasz `search_presidents` —
model nie "zna" tych danych z treningu, ale może je **odpytać** gdy potrzebuje.

To jest fundament architektury **RAG** (Retrieval Augmented Generation), którą poznamy na następnych zajęciach,
oraz **enterprise AI assistants** — wewnętrznych asystentów firmowych.

</div>

## 9. Combo: Function Calling + Structured Output

W łańcuchu function calli argumenty narzędzi są strukturalne (protokół `tools=`),
ale **końcowa odpowiedź** LLM-a to wolny tekst. Co jeśli chcemy ją ustrukturyzować?

<div style="background:#fff3cd; border-left:4px solid #ffc107; padding:12px; border-radius:4px;">

**Combo pattern (użyjemy go w ćwiczeniu 4):**
1. **Function Calling** (`client` + `tools=`) → zbieramy dane z narzędzi
2. **Structured Output** (`instructor_client` + `response_model=`) → formatujemy końcowy werdykt

Dzięki temu końcowa odpowiedź to nie tekst do parsowania, ale obiekt z polami: `verdict`, `confidence`, `source`.
</div>

Combo pattern w praktyce — funkcja `verify_claim()`:

In [29]:
def verify_claim(claim: str) -> "FactCheck":
    """
    Combo pattern:
      Krok 1 (FC): szukaj dowodów — najpierw search_presidents (LLM strukturyzuje query),
                    a jeśli nie znajdzie → FC z WSZYSTKIMI narzędziami (Wikipedia, web, itp.)
      Krok 2 (Structured Output): instructor formatuje werdykt w FactCheck
    """

    # ── Krok 1a: FC → search_presidents (nasza baza z fikcyjnymi faktami!) ──
    pres_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content":
             "Znajdź informacje o tym twierdzeniu w bazie prezydentów. "
             "ZAWSZE użyj narzędzia search_presidents — nie odpowiadaj z pamięci."},
            {"role": "user", "content": claim}
        ],
        tools=[t for t in tools_definition if t["function"]["name"] == "search_presidents"],
        temperature=0.1,
    )
    pres_msg = pres_response.choices[0].message

    evidence = "Nie znaleziono"
    source = "brak"
    if pres_msg.tool_calls:
        tc = pres_msg.tool_calls[0]
        func_args = json.loads(tc.function.arguments)
        evidence = search_presidents(**func_args)
        source = "search_presidents"

    # ── Krok 1b: Jeśli baza prezydentów nie pomogła → FC ze wszystkimi narzędziami ──
    if "Nie znaleziono" in evidence:
        full_response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content":
                 "Znajdź dowody na temat tego twierdzenia. Użyj dostępnych narzędzi. "
                 "ZAWSZE użyj narzędzia — nie odpowiadaj z pamięci."},
                {"role": "user", "content": claim}
            ],
            tools=tools_definition,
            temperature=0.1,
        )
        full_msg = full_response.choices[0].message
        if full_msg.tool_calls:
            tc = full_msg.tool_calls[0]
            func_name = tc.function.name
            func_args = json.loads(tc.function.arguments)
            evidence = AVAILABLE_TOOLS.get(func_name, lambda **kw: "Nieznane narzędzie")(**func_args)
            source = func_name

    # ── Krok 2: Structured Output — instructor formatuje werdykt ──
    # instructor wymusza format FactCheck. Hint o PŁASKIM JSONie to workaround
    # na mniejsze modele (np. Gemma) które owijają JSON w "properties".
    return instructor_client.chat.completions.create(
        model=MODEL_NAME,
        response_model=FactCheck,
        messages=[
            {"role": "system", "content":
             "Jesteś weryfikatorem faktów. Na podstawie DOWODÓW oceń twierdzenie. "
             "Jeśli dowody nie wystarczają, powiedz 'nie da się zweryfikować'. Bądź uczciwy. "
             "Odpowiedz PŁASKIM JSON-em — NIE owijaj w 'properties'."},
            {"role": "user", "content":
             f"Twierdzenie: {claim}\n\nDowody (źródło: {source}):\n{evidence}"}
        ],
    )

print("Funkcja verify_claim() gotowa — potrzebuje modelu FactCheck (ćwiczenie 4).")

Funkcja verify_claim() gotowa — potrzebuje modelu FactCheck (ćwiczenie 4).


### Ćwiczenie 4: FactCheck — zweryfikuj twierdzenie strukturalnie

Funkcja `verify_claim()` jest gotowa powyżej — brakuje jej tylko **modelu `FactCheck`**.

**Twoje zadanie:** Uzupełnij 5 pól Pydantic:
- `claim` — sprawdzane twierdzenie (str)
- `evidence` — znalezione dowody (str)
- `verdict` — werdykt: `Literal["prawda", "fałsz", "nie da się zweryfikować"]`
- `confidence` — pewność 0-1: `Field(..., ge=0, le=1)`
- `source` — skąd pochodzi dowód (str)

`instructor` wymusza, żeby LLM zwrócił obiekt dokładnie w formacie Twojego modelu.
Jeśli LLM pominie pole, instructor automatycznie ponawia zapytanie z feedbackiem!

In [30]:
# Ćwiczenie 4: Uzupełnij model FactCheck

class FactCheck(BaseModel):
    claim: str = Field(..., description="Sprawdzane twierdzenie")
    evidence: str = Field(..., description="Znalezione dowody potwierdzające lub obalające")
    verdict: Literal["prawda", "fałsz", "nie da się zweryfikować"] = Field(
        ..., description="Werdykt na podstawie dowodów"
    )
    confidence: float = Field(..., ge=0, le=1, description="Pewność werdyktu (0=zgaduję, 1=pewny)")
    source: str = Field(..., description="Skąd pochodzą dowody, np. 'baza prezydentów', 'Wikipedia'")

###### <span style="color: #c17f24;">Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Użyj `Literal["prawda", "fałsz", "nie da się zweryfikować"]` dla verdict, `Field(..., ge=0, le=1)` dla confidence. Każde pole potrzebuje `Field(..., description="...")`. Funkcja `verify_claim()` jest gotowa — wystarczy uzupełnić model.

###### <span style="color: #5a8a6a;">Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [31]:
# Ćwiczenie 4 — rozwiązanie (tylko model — verify_claim jest gotowa powyżej)

class FactCheck(BaseModel):
    claim: str = Field(..., description="Sprawdzane twierdzenie")
    evidence: str = Field(..., description="Znalezione dowody potwierdzające lub obalające")
    verdict: Literal["prawda", "fałsz", "nie da się zweryfikować"] = Field(
        ..., description="Werdykt na podstawie dowodów"
    )
    confidence: float = Field(..., ge=0, le=1, description="Pewność werdyktu (0=zgaduję, 1=pewny)")
    source: str = Field(..., description="Skąd pochodzą dowody, np. 'baza prezydentów', 'Wikipedia'")

##### 

In [32]:
# --- TEST ---
if instructor_client:
    twierdzenia = [
        "Lech Wałęsa dostał Pokojową Nagrodę Nobla w 1983 roku",
        "Aleksander Kwaśniewski posiadał popiersie Kleopatry",
        "Kraków jest stolicą Polski",
    ]
    for t in twierdzenia:
        try:
            fc = verify_claim(t)
            print(f"Twierdzenie: {t}")
            print(f"  Werdykt:  {fc.verdict} (pewność: {fc.confidence:.0%})")
            print(f"  Dowody:   {fc.evidence[:100]}...")
            print(f"  Źródło:   {fc.source}")
        except Exception as e:
            print(f"Twierdzenie: {t}")
            print(f"  ⚠️ Model nie wygenerował poprawnego JSON-a: {str(e)[:150]}")
            print("  (Mniejsze modele czasem owijają JSON w 'properties' — spróbuj z większym modelem)")
        print()
else:
    print("instructor_client niedostępny — pomiń to ćwiczenie.")

Twierdzenie: Lech Wałęsa dostał Pokojową Nagrodę Nobla w 1983 roku
  Werdykt:  prawda (pewność: 100%)
  Dowody:   Lech Wałęsa jest wymieniony jako 'Laureat Pokojowej Nagrody Nobla (1983)' w dostarczonych dowodach....
  Źródło:   search_presidents



Twierdzenie: Aleksander Kwaśniewski posiadał popiersie Kleopatry
  Werdykt:  prawda (pewność: 100%)
  Dowody:   W swoim gabinecie prezydenckim trzymał miniaturowe popiersie Kleopatry — podarunek od ambasadora Egi...
  Źródło:   search_presidents



Twierdzenie: Kraków jest stolicą Polski
  Werdykt:  fałsz (pewność: 100%)
  Dowody:   Stolica Polski – główna siedziba administracyjna Polski, której funkcję pełni obecnie Warszawa. Pier...
  Źródło:   Wikipedia



<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:14px; border-radius:4px;">

**Co się tu wydarzyło?**

1. **Function Calling** pobrał dowody — z bazy prezydentów, Wikipedii lub weba (narzędzia `search_presidents`, `search_wikipedia`, `search_web`)
2. **Instructor** (Structured Output) wziął te dowody i wyciągnął z nich ustrukturyzowany werdykt (`FactCheck`)
3. Gdyby LLM pominął pole — instructor złapałby `ValidationError` i kazał spróbować ponownie

To jest wzorzec **ETL z LLM-em**: Extract (narzędzia) → Transform (instructor) → Load (obiekt Pydantic).

**Kiedy to stosować?**
- Gdy źródło danych zwraca **nieustrukturyzowany tekst** (Wikipedia, web search, PDF, email...)
- A Ty potrzebujesz **konkretnych pól** do dalszego przetwarzania
- I chcesz **gwarancję formatu** (instructor z retry)

</div>

###### 

## 10. Pętla agentowa — LLM który myśli i działa

Do tej pory obsługiwaliśmy **jedno** wywołanie narzędzia.
Ale co jeśli LLM potrzebuje **kilku** narzędzi po kolei?

Np. pytanie: *"Sprawdź w bazie prezydentów ile lat miał Kwaśniewski gdy został prezydentem i oblicz ile to dni"*
wymaga:
1. Wyszukania w bazie prezydentów
2. Obliczenia matematycznego

To jest **pętla agentowa** — LLM wielokrotnie wywołuje narzędzia aż znajdzie odpowiedź.

```
while LLM chce wywołać narzędzie:
    1. LLM wybiera narzędzie i argumenty
    2. Nasz kod wywołuje funkcję
    3. Wynik wraca do LLM-a
    4. LLM decyduje: potrzebuję jeszcze? → wróć do 1.
                     mam wszystko? → generuję odpowiedź
```

In [33]:
def agent(question, max_steps=6):
    if not client:
        print("LLM niedostępny.")
        return

    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT + " Możesz wywołać wiele narzędzi po kolei."},
        {"role": "user", "content": question}
    ]

    print(f"Pytanie: {question}")
    print(f"{'─'*60}")

    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools_definition, temperature=0.1
        )

        msg = response.choices[0].message

        print_reasoning(msg)

        content = clean_content(msg)
        if content and msg.tool_calls:
            print(f"  💬 LLM mówi: {content[:300]}")
            print()

        if not msg.tool_calls:
            print(f"\n  Krok {step+1}: LLM generuje odpowiedź")
            print(f"  {'─'*50}")
            print()
            display(Markdown(content or ""))
            return content

        messages.append(msg)
        for tool_call in msg.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)

            print(f"\n  Krok {step+1}: LLM wywołuje → {func_name}({func_args})")

            try:
                if func_name in AVAILABLE_TOOLS:
                    result = AVAILABLE_TOOLS[func_name](**func_args)
                else:
                    result = f"Nieznane narzędzie: {func_name}"
            except Exception as e:
                result = f"Błąd wywołania {func_name}: {e}"

            display_result = result[:120] + "..." if len(result) > 120 else result
            print(f"           Wynik ← \"{display_result}\"")

            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    print(f"\n  (Osiągnięto limit {max_steps} kroków)")

print("Agent gotowy! Użyj: agent('twoje pytanie')")

Agent gotowy! Użyj: agent('twoje pytanie')


In [34]:
if client:
    print("Test 1: Prezydenci + obliczenie")
    print("═"*60)
    agent("Ile lat miał Aleksander Kwaśniewski gdy skończył kadencję? Oblicz ile to w przybliżeniu dni.")

    print("\n\nTest 2: Pogoda + kalkulator")
    print("═"*60)
    agent("Jaka jest temperatura we Wrocławiu? Przelicz ją na stopnie Fahrenheita.")

    print("\n\nTest 3: Prezydenci + pogoda")
    print("═"*60)
    agent("W jakim mieście urodził się Andrzej Duda? Jaka jest tam teraz pogoda?")

Test 1: Prezydenci + obliczenie
════════════════════════════════════════════════════════════
Pytanie: Ile lat miał Aleksander Kwaśniewski gdy skończył kadencję? Oblicz ile to w przybliżeniu dni.
────────────────────────────────────────────────────────────


  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking for two things:
         *   The age of Aleksander Kwaśniewski when he finished his term as president.
         *   An approximate calculation of how many days that age represents (this phrasing is slightly ambiguous, but it likely means calculating the number of days in his lifespan up to that point).
     
     2. **Identify Necessary Tools:**
         *   To find information about Aleksander Kwaśniewski (specifically his age at the end of his term), the `search_president


  Krok 1: LLM wywołuje → search_presidents({'query': 'Aleksander Kwaśniewski wiek koniec kadencji'})
           Wynik ← "### Aleksander Kwaśniewski
  - kadencja: 1995–2005
  - lata życia: ur. 1954
  - miejsce urodzenia: Białogard
  - wiek w ..."


  🧠 Tok myślenia (reasoning):
     The user wants to know two things based on the previous search result:
     1.  How old Aleksander Kwaśniewski was when his term ended.
     2.  Approximately how many days that age corresponds to (this part of the request seems slightly ambiguous: "Oblicz ile to w przybliżeniu dni" - does it mean the number of days he lived *during* his term, or convert his age into days?).
     
     From the previous tool output:
     *   Aleksander Kwaśniewski's term was from 1995–2005.
     *   His life years: born in 1954.
     *   The outp


  Krok 2: LLM generuje odpowiedź
  ──────────────────────────────────────────────────



Aleksander Kwaśniewski urodził się w 1954 roku, a jego kadencja trwała do 2005 roku.

1.  **Wiek przy zakończeniu kadencji:**
    $2005 - 1954 = 51$ lat.

2.  **Długość kadencji w dniach:**
    Zgodnie z informacjami, jego kadencja trwała **3653 dni**.

*(Jeśli miałeś na myśli przeliczenie jego wieku (51 lat) na dni, to jest to w przybliżeniu $51 \times 365.25 \approx 18627$ dni).*



Test 2: Pogoda + kalkulator
════════════════════════════════════════════════════════════
Pytanie: Jaka jest temperatura we Wrocławiu? Przelicz ją na stopnie Fahrenheita.
────────────────────────────────────────────────────────────


...

  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking two questions:
         a) In which city was Andrzej Duda born?
         b) What is the current weather in that city?
     
     2. **Identify Necessary Tools:**
         * To find where Andrzej Duda was born, I need a tool that can search for biographical information. `search_wikipedia` or `search_web` are suitable. Since this is a specific biographical fact, Wikipedia is usually the best starting point for general knowledge.
         * To find the weather in that city, I need 


  Krok 1: LLM wywołuje → search_wikipedia({'query': 'Andrzej Duda miejsce urodzenia'})


           Wynik ← "Andrzej Duda

Andrzej Sebastian Duda (ur. 16 maja 1972 w Krakowie) – polski prawnik i polityk, doktor nauk prawnych. Pre..."



  Krok 2: LLM wywołuje → get_weather({'city': 'Kraków'})
           Wynik ← "Kraków: 15°C, Bezchmurnie, wilgotność 48%, dane z 2026-05-10 08:38 PM (wttr.in)"


  🧠 Tok myślenia (reasoning):
     The user asked two questions:
     1. In which city was Andrzej Duda born? (Answered by `search_wikipedia`)
     2. What is the weather like there now? (Answered by `get_weather` for Kraków)
     
     The first tool call (`search_wikipedia`) returned that Andrzej Duda was born in **Kraków**.
     The second tool call (`get_weather`) returned the current weather for **Kraków**: 15°C, Bezchmurnie, wilgotność 48%, dane z 2026-05-10 08:38 PM.
     
     I have all the necessary information to answer the user's request completely. I 


  Krok 3: LLM generuje odpowiedź
  ──────────────────────────────────────────────────



Andrzej Duda urodził się w **Krakowie**.

Obecna pogoda w Krakowie to: 15°C, bezchmurnie, wilgotność 48% (dane z 2026-05-10 20:38).

### Ćwiczenie 5: Zadaj agentowi trudne pytanie

Wymyśl pytanie, które wymaga użycia **dwóch lub więcej** narzędzi.
Teraz masz do dyspozycji: pogodę, kalkulator, prezydentów, Wikipedię, i web search.

Obserwuj jak agent krok po kroku dochodzi do odpowiedzi.

In [35]:
# Ćwiczenie 5: Twoje pytanie do agenta

MOJE_PYTANIE = "Ile lat miał Aleksander Kwaśniewski gdy skończył kadencję? Oblicz ile to w przybliżeniu dni."

if MOJE_PYTANIE is ...:
    print("\033[1;31m⬆️ Wpisz swoje pytanie powyżej! Zamień ... na string.\033[0m")
elif client:
    agent(MOJE_PYTANIE)

Pytanie: Ile lat miał Aleksander Kwaśniewski gdy skończył kadencję? Oblicz ile to w przybliżeniu dni.
────────────────────────────────────────────────────────────


  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking for two things:
         a. The age of Aleksander Kwaśniewski when he finished his term (kadencja).
         b. An approximate calculation of how many days that age represents.
     
     2. **Identify Necessary Tools:**
         * To find information about Aleksander Kwaśniewski (specifically his age at the end of his term), the `search_presidents` tool is appropriate.
         * To perform the calculation (age in years to days), the `calculate` tool is needed.
     
     3. **Step 1: Ge


  Krok 1: LLM wywołuje → search_presidents({'query': 'Aleksander Kwaśniewski wiek koniec kadencji'})
           Wynik ← "### Aleksander Kwaśniewski
  - kadencja: 1995–2005
  - lata życia: ur. 1954
  - miejsce urodzenia: Białogard
  - wiek w ..."


  🧠 Tok myślenia (reasoning):
     The user wants to know two things based on the previous search result:
     1.  How old Aleksander Kwaśniewski was when he finished his term.
     2.  Approximately how many days that age corresponds to (this part is slightly ambiguous, but likely means the duration of his term in days, or perhaps converting his age into a number of days).
     
     From the previous tool output:
     *   **Aleksander Kwaśniewski:** Kadencja 1995–2005.
     *   **Długość kadencji:** 3653 dni.
     
     The search result explicitly states the **durat


  Krok 2: LLM wywołuje → calculate({'expression': '51 * 365.25'})
           Wynik ← "51 * 365.25 = 18627.75"


  🧠 Tok myślenia (reasoning):
     The user asked for two things:
     1.  How old was Aleksander Kwaśniewski when he finished his term?
     2.  Calculate approximately how many days that is.
     
     From the first tool call (`search_presidents`), I found:
     *   Aleksander Kwaśniewski's birth year: 1954.
     *   His term was from 1995–2005.
     
     To find his age at the end of the term (2005):
     Age = 2005 - 1954 = 51 years.
     
     The second part of the request is to calculate this in days, approximately.
     I used a second tool call (`calculate`) with the expression


  Krok 3: LLM generuje odpowiedź
  ──────────────────────────────────────────────────



Aleksander Kwaśniewski miał **51 lat**, gdy skończył kadencję (w 2005 roku).

Przybliżona liczba dni, jaką to stanowi, wynosi około **18 628 dni** (51 lat \* 365,25 dnia/rok).

###### <span style="color: #c17f24;">Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Przykłady pytań łączących narzędzia:
- *"Sprawdź na Wikipedii kto wynalazł telefon i oblicz ile lat temu to było"*
- *"Jaka jest pogoda w Krakowie i ile mieszkańców ma Kraków? Oblicz ile osób przypada na 1 stopień temperatury"*
- *"Znajdź na Wikipedii wysokość Mont Blanc i przelicz ją na stopy (1 stopa = 0.3048 m)"*
- *"Jakie hobby ma Andrzej Duda wg bazy prezydentów? Sprawdź na Wikipedii czy to prawda."*

###### 

### Ćwiczenie 6 (bonus): Zbuduj własnego asystenta

Wybierz **temat** i stwórz mini-asystenta z własnymi narzędziami:

| Temat | Pomysły na narzędzia |
|---|---|
| Asystent podróżniczy | pogoda + Wikipedia (miasta) + kalkulator (waluty) |
| Asystent filmowy | Wikipedia (filmy) + web search (recenzje) + kalkulator (oceny) |
| Asystent naukowy | Wikipedia + kalkulator + web search (artykuły) |

**Zadanie:**
1. Wybierz temat
2. Zdefiniuj system prompt dopasowany do tematu
3. Opcjonalnie: dodaj 1 nowe narzędzie specyficzne dla tematu
4. Przetestuj 3 pytaniami

In [36]:
# Ćwiczenie 6 (bonus): Twój asystent

TEMAT = "asystent podróżniczy"
SYSTEM_PROMPT = "Jesteś asystentem podróżniczym. Pomagasz planować wycieczki, sprawdzasz pogodę i populację miast. Odpowiadaj po polsku."

def my_agent(question):
    if not client:
        print("LLM niedostępny.")
        return

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question}
    ]

    print(f"[{TEMAT}] {question}")
    print(f"{'─'*60}")

    for step in range(5):
        response = client.chat.completions.create(
            model=MODEL_NAME, messages=messages, tools=tools_definition, temperature=0.3
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            print(f"\n  Odpowiedź: {msg.content}")
            return

        messages.append(msg)
        for tc in msg.tool_calls:
            fn = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"  Krok {step+1}: {fn}({args})")
            try:
                result = AVAILABLE_TOOLS.get(fn, lambda **kw: "Nieznane narzędzie")(**args)
            except Exception as e:
                result = f"Błąd: {e}"
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

if TEMAT == "..." or SYSTEM_PROMPT == "...":
    print("\033[1;31m⬆️ Uzupełnij TEMAT i SYSTEM_PROMPT powyżej!\033[0m")
elif client:
    my_agent("Twoje pierwsze pytanie testowe")

[asystent podróżniczy] Twoje pierwsze pytanie testowe
────────────────────────────────────────────────────────────



  Odpowiedź: Jestem gotów! Jak mogę Ci dzisiaj pomóc w planowaniu podróży, sprawdzeniu pogody, populacji miasta czy znalezieniu jakiejkolwiek innej informacji? 😊


###### <span style="color: #c17f24;">Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Dla asystenta podróżniczego: `SYSTEM_PROMPT = "Jesteś asystentem podróżniczym. Pomagasz planować wycieczki. Używaj Wikipedii do znalezienia informacji o miastach, pogody do sprawdzenia warunków, i kalkulatora do przeliczeń walutowych. Odpowiadaj po polsku, entuzjastycznie."`

###### 

## 11. Podsumowanie

### Co zrobiliśmy?

1. **Pydantic + instructor** — ustrukturyzowane dane, JSON Schema, walidacja + LLM odpowiada jako obiekt Pythona
2. **Function Calling vs. Structured Output** — dwa różne mechanizmy, dwa klienty, komplementarne role
3. **Narzędzia** — pogoda (prawdziwe API!), kalkulator, baza prezydentów, Wikipedia, web search
4. **Pod maską** — co dokładnie LLM widzi, jak wybiera funkcję, JSON Schema ręcznie vs. automatycznie
5. **Combo pattern** — FC zbiera dane z narzędzi → Structured Output formatuje końcowy werdykt (FactCheck!)
6. **Pętla agentowa** — LLM sam decyduje które narzędzia wywołać i w jakiej kolejności

### Tak działają prawdziwe produkty AI

| Produkt | Narzędzia | Pydantic / Structured Output |
|---|---|---|
| ChatGPT | Przeglądarka, DALL-E, Code Interpreter | Wewnętrznie (JSON mode) |
| Claude | Wyszukiwanie, analiza plików, MCP | Tool use + JSON Schema |
| GitHub Copilot | Czytanie/pisanie plików, terminal | Structured tool responses |
| Cursor / Windsurf | Edycja kodu, terminal, dokumentacja | Structured diffs |

Wszystkie działają na tej samej zasadzie: **LLM + zestaw narzędzi + pętla agentowa + strukturalne odpowiedzi**.

### Kluczowe wnioski

<div style="background:#e8f4f8; border-left:4px solid #2196F3; padding:12px; border-radius:4px;">

**1. Description decyduje o wszystkim** — LLM wybiera narzędzie na podstawie opisu, nie kodu (ćwiczenie 1)

**2. Pydantic to nie tylko walidacja** — automatycznie generuje JSON Schema, wymusza strukturę odpowiedzi narzędzi

**3. FC + Structured Output = combo** — FC zbiera dane z prawdziwych źródeł, instructor formatuje końcowy werdykt w obiekt z polami

**4. Agent jest tak dobry jak jego narzędzia** — fikcyjne fakty o prezydentach udowodniły, że LLM wierzy temu co mu podamy

**5. Defensive coding jest konieczny** — web search może nie zadziałać, Wikipedia może zwrócić disambiguation, pogoda może być niedostępna (fallback!)

</div>

## 12. Bonus: Chat z pamięcią konwersacji

`ask_with_tools()` traktuje każde pytanie osobno — nie pamięta poprzednich.
Co jeśli chcemy, żeby LLM pamiętał kontekst? Np.:

```
Ty:  "Jaka pogoda we Wrocławiu?"
LLM: "15°C, częściowe zachmurzenie"
Ty:  "A jaka populacja tego miasta?"       ← LLM musi pamiętać, że chodzi o Wrocław!
```

Rozwiązanie: **`messages` żyje poza funkcją** — każde pytanie widzi poprzednie.

In [37]:
def chat_with_tools(messages, question, verbose=True):
    """ask_with_tools z pamięcią — messages jest listą modyfikowaną in-place."""
    if not client:
        print("LLM niedostępny.")
        return None

    # Przy pierwszym pytaniu dodaj system prompt
    if not messages:
        messages.append({"role": "system", "content": DEFAULT_SYSTEM_PROMPT})

    messages.append({"role": "user", "content": question})

    response = client.chat.completions.create(
        model=MODEL_NAME, messages=messages, tools=tools_definition, temperature=0.1
    )
    assistant_msg = response.choices[0].message

    print_reasoning(assistant_msg)
    content = clean_content(assistant_msg)

    if not assistant_msg.tool_calls:
        if verbose:
            print(f"  Narzędzie: BRAK (LLM odpowiedział sam)\n")
            display(Markdown(content or ""))
        messages.append({"role": "assistant", "content": content})
        return content

    messages.append(assistant_msg)
    n_calls = len(assistant_msg.tool_calls)
    for i, tool_call in enumerate(assistant_msg.tool_calls, 1):
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        label = f"[{i}/{n_calls}] " if n_calls > 1 else ""
        if verbose:
            print(f"  {label}Narzędzie: {func_name}({func_args})")
        result = AVAILABLE_TOOLS.get(func_name, lambda **kw: "Nieznane narzędzie")(**func_args)
        if verbose:
            print(f"  {label}Wynik:     {result[:150]}")
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    final = client.chat.completions.create(
        model=MODEL_NAME, messages=messages, temperature=0.1
    )
    final_answer = clean_content(final.choices[0].message)
    messages.append({"role": "assistant", "content": final_answer})

    if verbose:
        print()
        display(Markdown(final_answer or ""))
    return final_answer

print("Funkcja chat_with_tools() gotowa!")

Funkcja chat_with_tools() gotowa!


In [38]:
# Testujemy! Każde kolejne pytanie pamięta kontekst poprzednich.

if client:
    history = []

    print("─── Pytanie 1 ───")
    chat_with_tools(history, "Jaka jest pogoda we Wrocławiu?")

    print("\n─── Pytanie 2 (LLM pamięta 'Wrocław') ───")
    chat_with_tools(history, "A jaka populacja tego miasta?")

    print("\n─── Pytanie 3 (kontynuacja) ───")
    chat_with_tools(history, "Znajdź coś o tym mieście na Wikipedii")

    print(f"\n{'═'*60}")
    print(f"Historia konwersacji: {len(history)} wiadomości")

─── Pytanie 1 ───


  🧠 Tok myślenia (reasoning):
     1. **Analyze the user's request:** The user is asking "Jaka jest pogoda we Wrocławiu?" (What is the weather in Wrocław?).
     
     2. **Examine available tools:** I have several tools: `get_weather`, `calculate`, `search_presidents`, `get_population`, `search_wikipedia`, and `search_web`.
     
     3. **Identify the relevant tool:** The `get_weather` tool is specifically designed to check the current weather in a given city.
     
     4. **Determine required arguments:** The `get_weather` tool requires a `city` argument 

  Narzędzie: get_weather({'city': 'Wrocław'})


  Wynik:     Wrocław: 19°C, Słonecznie, wilgotność 43%, dane z 2026-05-10 07:52 PM (wttr.in)


Obecnie we Wrocławiu jest **19°C**, a pogoda jest **słoneczna**. Wilgotność wynosi 43%. (Dane z 10 maja 2026, godz. 19:52).


─── Pytanie 2 (LLM pamięta 'Wrocław') ───


...


─── Pytanie 3 (kontynuacja) ───


  🧠 Tok myślenia (reasoning):
     1. **Analyze the Request:** The user is asking to "Find something about this city on Wikipedia."
     2. **Identify the Context:** The previous turns established that "this city" refers to Wrocław.
     3. **Determine the Necessary Tool:** The `search_wikipedia` tool is designed for retrieving information from Wikipedia based on a query.
     4. **Formulate the Tool Call:** The query should be "Wrocław" to get general information about the city.
     5. **Construct the JSON:** `{"tool_name": "search_wikipedia", "pa

  Narzędzie: search_wikipedia({'query': 'Wrocław'})


  Wynik:     Wrocław

Wrocław (łac. Vratislavia, niem. Breslau, dś. Brassel, Gruß-Brassel, cz. Vratislav) – miasto na prawach powiatu w południowo-zachodniej Polsc


Oto krótki opis Wrocławia z Wikipedii:

**Wrocław** to miasto na prawach powiatu w południowo-zachodniej Polsce, siedziba władz województwa dolnośląskiego i powiatu wrocławskiego. Położone jest w Europie Środkowej, na Nizinie Śląskiej, nad Odrą i czterema jej dopływami. Jest historyczną stolicą Dolnego Śląska, a także całego Śląska.

Jest głównym miastem aglomeracji wrocławskiej i trzecim pod względem liczby ludności miastem w Polsce (po Warszawie i Krakowie), liczącym 672 601 mieszkańców.

*(Źródło: Wikipedia)*


════════════════════════════════════════════════════════════
Historia konwersacji: 13 wiadomości


## 13. 🚀 Bonus: Chat UI — jak ChatGPT, ale Twój!

Wszystkie funkcje które napisaliśmy (`ask_with_tools`, `agent`, `chat_with_tools`) działają w notebooku.
Ale co gdyby nasz agent wyglądał **jak prawdziwy ChatGPT**?

Jedna linijka kodu — i otwiera się przeglądarka z:
- 💬 **dymkami wiadomości** (Ty po prawej, model po lewej)
- 🧠 **collapsible reasoning** (tok myślenia modelu)
- 🔧 **tool calls z wynikami** (widać co model wywołał)
- 📝 **pamięcią konwersacji** (multi-turn jak ChatGPT)
- 🔄 **agent loop** (model może wywołać wiele narzędzi po kolei)

Pod spodem to dokładnie ten sam kod co pisaliśmy — tylko opakowany w [Gradio](https://gradio.app/).

In [39]:
# SKIPPED: from utils import ensure_package
print('Gradio launch skipped')

Gradio launch skipped


<div style="background:#e8f4f8; border-left:4px solid #17a2b8; padding:14px; border-radius:4px;">

**Co się tu wydarzyło?**

Cały nasz warsztat — od `calculate()` po `agent()` — zamknęliśmy w jednym pliku `chat_ui.py` (~150 linii).
Biblioteka [Gradio](https://gradio.app/) daje nam profesjonalny interfejs webowy **za darmo**.

To jest dokładnie tak jak buduje się narzędzia AI w firmach:
1. **Backend** = Function Calling (to co pisaliśmy)
2. **Frontend** = Gradio / Streamlit / React
3. **Razem** = pełna aplikacja AI

Twój lokalny LLM + nasze narzędzia = **Twój własny ChatGPT** 🎉

</div>